In [ ]:
# ============================================================
# 01. SELECCIÓN Y VALIDACIÓN DE BASE DE MODELADO
# Nearshoring Project / Market Opportunity Analytics
#
# Input esperado:
# df ya cargado en memoria desde el bloque anterior
#
# Outputs:
# 01_modeling_base_selected_columns.csv
# 02_modeling_base_validation_report.csv
# 03_modeling_base_missing_summary.csv
# 04_modeling_base_numeric_profile.csv
# 05_modeling_base_feature_correlations.csv
# 06_modeling_base_duplicate_keys.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

modeling_path.mkdir(parents=True, exist_ok=True)

print("Modeling output path:")
print(modeling_path)


# ------------------------------------------------------------
# 2. Columnas esperadas
# ------------------------------------------------------------

id_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key"
]

feature_cols = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",
    "denue_manufacturing_establishments",

    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments",

    "b2b_support_per_100_manufacturing_establishments",
    "b2b_medium_large_per_100_manufacturing_establishments",
    "b2b_large_per_100_manufacturing_establishments",
    "b2b_support_per_1000_manufacturing_workers",
    "b2b_medium_large_per_1000_manufacturing_workers",
    "b2b_large_per_1000_manufacturing_workers"
]

audit_cols = [
    "has_denue_match",
    "has_b2b_support",
    "has_b2b_medium_large",
    "has_b2b_large",
    "b2b_only_micro_small"
]

previous_output_cols = [
    "industrial_capacity_score",
    "industrial_capacity_percentile",
    "rank_industrial_capacity_within_year",
    "industrial_capacity_group",
    "b2b_support_percentile",
    "b2b_medium_large_percentile",
    "b2b_large_percentile",
    "b2b_support_supply_score",
    "b2b_support_supply_percentile",
    "high_industrial_capacity",
    "high_b2b_support",
    "high_b2b_medium_large",
    "high_b2b_large",
    "industrial_b2b_typology_broad",
    "industrial_b2b_typology_scaled"
]


# ------------------------------------------------------------
# 3. Verificar columnas existentes
# ------------------------------------------------------------

expected_cols = id_cols + feature_cols + audit_cols + previous_output_cols

existing_cols = [col for col in expected_cols if col in df.columns]
missing_expected_cols = [col for col in expected_cols if col not in df.columns]

existing_feature_cols = [col for col in feature_cols if col in df.columns]
missing_feature_cols = [col for col in feature_cols if col not in df.columns]

print("Columnas esperadas existentes:", len(existing_cols))
print("Features existentes:", len(existing_feature_cols))

if missing_expected_cols:
    print("\nColumnas esperadas no encontradas:")
    for col in missing_expected_cols:
        print("-", col)


# ------------------------------------------------------------
# 4. Crear base seleccionada
# ------------------------------------------------------------

modeling_base = df[existing_cols].copy()

# Homologar llaves
modeling_base["year"] = pd.to_numeric(
    modeling_base["year"],
    errors="coerce"
).astype("Int64")

modeling_base["entidad_id"] = (
    modeling_base["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

modeling_base["municipio_id"] = (
    modeling_base["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

modeling_base["geo_key"] = (
    modeling_base["entidad_id"] + modeling_base["municipio_id"]
)

# Asegurar numéricos en features
for col in existing_feature_cols:
    modeling_base[col] = pd.to_numeric(
        modeling_base[col],
        errors="coerce"
    )

print("\nBase seleccionada:")
print(f"Filas: {modeling_base.shape[0]:,}")
print(f"Columnas: {modeling_base.shape[1]:,}")

display(modeling_base.head())


# ------------------------------------------------------------
# 5. Validación de llave year × geo_key
# ------------------------------------------------------------

duplicate_keys = (
    modeling_base[
        modeling_base.duplicated(
            subset=["year", "geo_key"],
            keep=False
        )
    ]
    .sort_values(["year", "geo_key"])
    .copy()
)

n_duplicate_rows = len(duplicate_keys)

key_is_unique = n_duplicate_rows == 0


# ------------------------------------------------------------
# 6. Cobertura por año
# ------------------------------------------------------------

coverage_by_year = (
    modeling_base
    .groupby("year", dropna=False)
    .agg(
        n_rows=("geo_key", "size"),
        n_unique_municipalities=("geo_key", "nunique"),
        n_entities=("entidad_id", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. Missing values
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "column": modeling_base.columns,
    "missing_count": modeling_base.isna().sum().values,
    "missing_share": modeling_base.isna().mean().values
}).sort_values(
    ["missing_share", "missing_count"],
    ascending=[False, False]
)


# ------------------------------------------------------------
# 8. Ceros por feature
# ------------------------------------------------------------

zero_summary = []

for col in existing_feature_cols:
    s = modeling_base[col]
    zero_count = (s == 0).sum()
    zero_summary.append({
        "column": col,
        "zero_count": zero_count,
        "zero_share": zero_count / len(modeling_base)
    })

zero_summary = pd.DataFrame(zero_summary)


# ------------------------------------------------------------
# 9. Perfil numérico básico
# ------------------------------------------------------------

numeric_profile = (
    modeling_base[existing_feature_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

numeric_profile = numeric_profile.merge(
    zero_summary,
    left_on="feature",
    right_on="column",
    how="left"
).drop(columns=["column"])


# ------------------------------------------------------------
# 10. Correlaciones entre features
# ------------------------------------------------------------

feature_correlations = (
    modeling_base[existing_feature_cols]
    .corr(method="pearson")
    .reset_index()
    .rename(columns={"index": "feature"})
)


# ------------------------------------------------------------
# 11. Reporte general de validación
# ------------------------------------------------------------

validation_items = [
    {
        "check": "n_rows",
        "value": modeling_base.shape[0],
        "status": "info"
    },
    {
        "check": "n_columns",
        "value": modeling_base.shape[1],
        "status": "info"
    },
    {
        "check": "n_features_expected",
        "value": len(feature_cols),
        "status": "info"
    },
    {
        "check": "n_features_existing",
        "value": len(existing_feature_cols),
        "status": "ok" if len(missing_feature_cols) == 0 else "warning"
    },
    {
        "check": "year_geo_key_unique",
        "value": key_is_unique,
        "status": "ok" if key_is_unique else "warning"
    },
    {
        "check": "duplicate_key_rows",
        "value": n_duplicate_rows,
        "status": "ok" if n_duplicate_rows == 0 else "warning"
    },
    {
        "check": "n_years",
        "value": modeling_base["year"].nunique(dropna=True),
        "status": "info"
    },
    {
        "check": "years_available",
        "value": ", ".join(
            modeling_base["year"].dropna().astype(str).sort_values().unique()
        ),
        "status": "info"
    },
    {
        "check": "missing_expected_columns",
        "value": ", ".join(missing_expected_cols) if missing_expected_cols else "",
        "status": "ok" if len(missing_expected_cols) == 0 else "warning"
    },
    {
        "check": "missing_feature_columns",
        "value": ", ".join(missing_feature_cols) if missing_feature_cols else "",
        "status": "ok" if len(missing_feature_cols) == 0 else "warning"
    }
]

validation_report = pd.DataFrame(validation_items)


# ------------------------------------------------------------
# 12. Guardar outputs
# ------------------------------------------------------------

modeling_base.to_csv(
    modeling_path / "01_modeling_base_selected_columns.csv",
    index=False,
    encoding="utf-8-sig"
)

validation_report.to_csv(
    modeling_path / "02_modeling_base_validation_report.csv",
    index=False,
    encoding="utf-8-sig"
)

missing_summary.to_csv(
    modeling_path / "03_modeling_base_missing_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_profile.to_csv(
    modeling_path / "04_modeling_base_numeric_profile.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_correlations.to_csv(
    modeling_path / "05_modeling_base_feature_correlations.csv",
    index=False,
    encoding="utf-8-sig"
)

duplicate_keys.to_csv(
    modeling_path / "06_modeling_base_duplicate_keys.csv",
    index=False,
    encoding="utf-8-sig"
)

coverage_by_year.to_csv(
    modeling_path / "07_modeling_base_coverage_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 13. Mostrar resultados clave
# ------------------------------------------------------------

print("\nVALIDATION REPORT")
display(validation_report)

print("\nCOVERAGE BY YEAR")
display(coverage_by_year)

print("\nMISSING SUMMARY")
display(missing_summary.head(20))

print("\nNUMERIC PROFILE")
display(numeric_profile.head(20))

print("\nArchivos guardados en:")
print(modeling_path)

Modeling output path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base
Columnas esperadas existentes: 53
Features existentes: 27

Base seleccionada:
Filas: 4,898
Columnas: 53


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,total_manufacturing_establishments,total_manufacturing_employment,total_manufacturing_value_added,total_manufacturing_income,...,b2b_medium_large_percentile,b2b_large_percentile,b2b_support_supply_score,b2b_support_supply_percentile,high_industrial_capacity,high_b2b_support,high_b2b_medium_large,high_b2b_large,industrial_b2b_typology_broad,industrial_b2b_typology_scaled
0,2018,15,México,106,Toluca,15106,4351.0,83598.0,97605.998,334675.563,...,0.991009,0.994279,0.973590,0.986514,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,3824.0,126640.0,77223.073,274960.549,...,0.995913,0.996731,0.980486,0.993870,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
2,2018,02,Baja California,004,Tijuana,02004,3670.0,270055.0,85526.588,222120.705,...,0.996731,0.995913,0.975429,0.989375,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
3,2018,22,Querétaro,014,Querétaro,22014,3363.0,103369.0,60661.916,229532.904,...,0.997548,0.997548,0.982938,0.996322,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,2046.0,70589.0,72451.993,201548.237,...,0.993053,0.995505,0.982274,0.995505,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub



VALIDATION REPORT


,check,value,status
0,n_rows,4898,info
1,n_columns,53,info
2,n_features_expected,27,info
3,n_features_existing,27,ok
4,year_geo_key_unique,True,ok
5,duplicate_key_rows,0,ok
6,n_years,2,info
7,years_available,"2018, 2023",info
8,missing_expected_columns,,ok
9,missing_feature_columns,,ok



COVERAGE BY YEAR


...


MISSING SUMMARY


,column,missing_count,missing_share
7,total_manufacturing_employment,368,0.075133
8,total_manufacturing_value_added,368,0.075133
9,total_manufacturing_income,368,0.075133
10,total_manufacturing_investment,368,0.075133
30,b2b_support_per_1000_manufacturing_workers,368,0.075133
31,b2b_medium_large_per_1000_manufacturing_workers,368,0.075133
32,b2b_large_per_1000_manufacturing_workers,368,0.075133
38,industrial_capacity_score,368,0.075133
39,industrial_capacity_percentile,368,0.075133
40,rank_industrial_capacity_within_year,368,0.075133



NUMERIC PROFILE


,feature,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,zero_count,zero_share
0,total_manufacturing_establishments,4898.0,247.871784,622.304482,1.000,1.00000,4.00000,21.000000,65.000000,208.000000,1026.600000,3061.690000,12015.000,0,0.000000
1,total_manufacturing_employment,4530.0,3018.688079,14862.541454,3.000,8.00000,16.00000,65.000000,217.000000,971.500000,10814.950000,66193.080000,364180.000,0,0.000000
2,total_manufacturing_value_added,4530.0,1894.451026,10348.258186,-11442.849,0.13829,0.50600,3.675750,15.993000,134.570250,6118.078200,51647.924600,226838.388,0,0.000000
3,total_manufacturing_income,4530.0,5773.935490,29826.572473,0.089,0.47329,1.33535,8.705000,37.518500,372.154500,18401.056500,163744.983070,497003.622,0,0.000000
4,total_manufacturing_investment,4530.0,97.521041,831.081055,-19194.283,-58.15498,-0.92725,0.001000,0.137000,2.370750,254.830100,2639.860460,21344.336,298,0.060841
5,denue_manufacturing_establishments,4898.0,232.965292,621.967606,0.000,1.00000,3.00000,17.000000,57.000000,182.000000,973.750000,3001.860000,11709.000,40,0.008167
6,denue_b2b_support_establishments,4898.0,97.418538,384.283393,0.000,0.00000,0.00000,3.000000,11.000000,40.000000,373.000000,1943.270000,7008.000,393,0.080237
7,denue_logistics_storage_establishments,4898.0,16.089628,61.977091,0.000,0.00000,0.00000,0.000000,2.000000,7.000000,61.000000,326.150000,932.000,1627,0.332176
8,denue_professional_technical_establishments,4898.0,43.234381,201.635179,0.000,0.00000,0.00000,0.000000,2.000000,13.000000,167.000000,905.270000,4022.000,1605,0.327685
9,denue_business_support_establishments,4898.0,38.094528,136.177591,0.000,0.00000,0.00000,2.000000,6.000000,19.000000,147.150000,682.120000,2440.000,555,0.113312



Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base


In [ ]:
# ============================================================
# 02. CREAR BASE MODEL-READY EXCLUYENDO MISSING EN FEATURES
# Nearshoring Project / Market Opportunity Analytics
#
# Input:
# modeling_base ya creada en memoria
#
# Outputs:
# 08_modeling_base_complete_cases.csv
# 09_modeling_base_excluded_missing_cases.csv
# 10_modeling_base_complete_case_report.csv
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Ruta de salida
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

modeling_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Confirmar features disponibles
# ------------------------------------------------------------

feature_cols = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",
    "denue_manufacturing_establishments",

    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments",

    "b2b_support_per_100_manufacturing_establishments",
    "b2b_medium_large_per_100_manufacturing_establishments",
    "b2b_large_per_100_manufacturing_establishments",
    "b2b_support_per_1000_manufacturing_workers",
    "b2b_medium_large_per_1000_manufacturing_workers",
    "b2b_large_per_1000_manufacturing_workers"
]

existing_feature_cols = [
    col for col in feature_cols
    if col in modeling_base.columns
]

missing_feature_cols = [
    col for col in feature_cols
    if col not in modeling_base.columns
]

print("Features existentes:", len(existing_feature_cols))
print("Features faltantes:", len(missing_feature_cols))

if missing_feature_cols:
    print("\nFeatures faltantes:")
    for col in missing_feature_cols:
        print("-", col)


# ------------------------------------------------------------
# 3. Crear bandera de casos completos
# ------------------------------------------------------------

modeling_base = modeling_base.copy()

modeling_base["is_complete_case_for_modeling"] = (
    modeling_base[existing_feature_cols]
    .notna()
    .all(axis=1)
)

modeling_base_complete = (
    modeling_base[
        modeling_base["is_complete_case_for_modeling"]
    ]
    .copy()
)

modeling_base_excluded = (
    modeling_base[
        ~modeling_base["is_complete_case_for_modeling"]
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Reporte de exclusión
# ------------------------------------------------------------

complete_case_report = pd.DataFrame([
    {
        "metric": "original_rows",
        "value": len(modeling_base)
    },
    {
        "metric": "complete_case_rows",
        "value": len(modeling_base_complete)
    },
    {
        "metric": "excluded_rows_due_to_missing_features",
        "value": len(modeling_base_excluded)
    },
    {
        "metric": "excluded_share",
        "value": len(modeling_base_excluded) / len(modeling_base)
    },
    {
        "metric": "n_features_used_for_complete_case_filter",
        "value": len(existing_feature_cols)
    }
])


# ------------------------------------------------------------
# 5. Cobertura por año antes/después
# ------------------------------------------------------------

coverage_original = (
    modeling_base
    .groupby("year")
    .agg(
        original_rows=("geo_key", "size"),
        original_municipalities=("geo_key", "nunique")
    )
    .reset_index()
)

coverage_complete = (
    modeling_base_complete
    .groupby("year")
    .agg(
        complete_rows=("geo_key", "size"),
        complete_municipalities=("geo_key", "nunique")
    )
    .reset_index()
)

coverage_report = coverage_original.merge(
    coverage_complete,
    on="year",
    how="left"
)

coverage_report["excluded_rows"] = (
    coverage_report["original_rows"] -
    coverage_report["complete_rows"]
)

coverage_report["excluded_share"] = (
    coverage_report["excluded_rows"] /
    coverage_report["original_rows"]
)


# ------------------------------------------------------------
# 6. Missing por columna en casos excluidos
# ------------------------------------------------------------

excluded_missing_summary = pd.DataFrame({
    "feature": existing_feature_cols,
    "missing_count_among_excluded": [
        modeling_base_excluded[col].isna().sum()
        for col in existing_feature_cols
    ],
    "missing_share_among_excluded": [
        modeling_base_excluded[col].isna().mean()
        for col in existing_feature_cols
    ]
}).sort_values(
    "missing_count_among_excluded",
    ascending=False
)


# ------------------------------------------------------------
# 7. Guardar outputs
# ------------------------------------------------------------

modeling_base_complete.to_csv(
    modeling_path / "08_modeling_base_complete_cases.csv",
    index=False,
    encoding="utf-8-sig"
)

modeling_base_excluded.to_csv(
    modeling_path / "09_modeling_base_excluded_missing_cases.csv",
    index=False,
    encoding="utf-8-sig"
)

complete_case_report.to_csv(
    modeling_path / "10_modeling_base_complete_case_report.csv",
    index=False,
    encoding="utf-8-sig"
)

coverage_report.to_csv(
    modeling_path / "11_modeling_base_complete_case_coverage_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)

excluded_missing_summary.to_csv(
    modeling_path / "12_modeling_base_excluded_missing_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 8. Mostrar resultados
# ------------------------------------------------------------

print("\nCOMPLETE CASE REPORT")
display(complete_case_report)

print("\nCOVERAGE BEFORE / AFTER")
display(coverage_report)

print("\nMISSING AMONG EXCLUDED CASES")
display(excluded_missing_summary)

print("\nArchivos guardados en:")
print(modeling_path)

Features existentes: 27
Features faltantes: 0

COMPLETE CASE REPORT


,metric,value
0,original_rows,4898.000000
1,complete_case_rows,4530.000000
2,excluded_rows_due_to_missing_features,368.000000
3,excluded_share,0.075133
4,n_features_used_for_complete_case_filter,27.000000



COVERAGE BEFORE / AFTER


,year,original_rows,original_municipalities,complete_rows,complete_municipalities,excluded_rows,excluded_share
0,2018,2447,2447,2245,2245,202,0.082550
1,2023,2451,2451,2285,2285,166,0.067727



MISSING AMONG EXCLUDED CASES


,feature,missing_count_among_excluded,missing_share_among_excluded
1,total_manufacturing_employment,368,1.0
2,total_manufacturing_value_added,368,1.0
3,total_manufacturing_income,368,1.0
4,total_manufacturing_investment,368,1.0
25,b2b_medium_large_per_1000_manufacturing_workers,368,1.0
24,b2b_support_per_1000_manufacturing_workers,368,1.0
26,b2b_large_per_1000_manufacturing_workers,368,1.0
7,denue_logistics_storage_establishments,0,0.0
0,total_manufacturing_establishments,0,0.0
5,denue_manufacturing_establishments,0,0.0



Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base


In [ ]:
# ============================================================
# 00. CONFIGURAR RUTAS Y CARGAR PANEL BASE
# Nearshoring Project / Market Opportunity Analytics
#
# Busca, en este orden:
# 1. municipality_industrial_b2b_opportunity_panel_curated
# 2. municipality_industrial_b2b_opportunity_panel
# 3. municipality_industrial_b2b_opportunity_panel_raw
#
# Outputs posteriores irán a:
# data/processed/modeling_base/
# ============================================================

from pathlib import Path
import pandas as pd
from google.colab import drive


# ------------------------------------------------------------
# 1. Montar Drive de forma segura
# ------------------------------------------------------------

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    drive.mount("/content/drive")
else:
    print("Google Drive ya está disponible.")


# ------------------------------------------------------------
# 2. Definir rutas del proyecto
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

processed_path = project_path / "data" / "processed"

modeling_path = processed_path / "modeling_base"
modeling_path.mkdir(parents=True, exist_ok=True)

print("Project path:")
print(project_path)

print("\nProcessed path:")
print(processed_path)

print("\nModeling output path:")
print(modeling_path)


# ------------------------------------------------------------
# 3. Buscar archivo base
# ------------------------------------------------------------

base_names_priority = [
    "municipality_industrial_b2b_opportunity_panel_curated",
    "municipality_industrial_b2b_opportunity_panel",
    "municipality_industrial_b2b_opportunity_panel_raw"
]

allowed_extensions = [".csv", ".xlsx", ".xls", ".parquet"]

matches = []

for base_name in base_names_priority:
    for path in project_path.rglob("*"):
        if path.is_file():
            if base_name.lower() == path.stem.lower() and path.suffix.lower() in allowed_extensions:
                matches.append({
                    "priority_name": base_name,
                    "path": path
                })

print("\nCoincidencias encontradas:")
print("=" * 100)

for i, item in enumerate(matches):
    print(f"{i} -> [{item['priority_name']}] {item['path']}")


# ------------------------------------------------------------
# 4. Validar que exista al menos un archivo
# ------------------------------------------------------------

if len(matches) == 0:
    raise FileNotFoundError(
        "No encontré ningún panel base dentro de Nearshoring_Project. "
        "Revisa si el archivo existe o si fue guardado con otro nombre."
    )


# ------------------------------------------------------------
# 5. Seleccionar el archivo de mayor prioridad
# ------------------------------------------------------------

selected_path = matches[0]["path"]

print("\nArchivo seleccionado:")
print(selected_path)


# ------------------------------------------------------------
# 6. Cargar archivo según extensión
# ------------------------------------------------------------

suffix = selected_path.suffix.lower()

if suffix == ".csv":
    df = pd.read_csv(
        selected_path,
        dtype={
            "year": str,
            "entidad_id": str,
            "municipio_id": str,
            "geo_key": str
        },
        low_memory=False
    )

elif suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(
        selected_path,
        dtype={
            "year": str,
            "entidad_id": str,
            "municipio_id": str,
            "geo_key": str
        }
    )

elif suffix == ".parquet":
    df = pd.read_parquet(selected_path)

else:
    raise ValueError(f"Extensión no soportada: {suffix}")


# ------------------------------------------------------------
# 7. Homologar llaves
# ------------------------------------------------------------

if "year" in df.columns:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

if "entidad_id" in df.columns:
    df["entidad_id"] = df["entidad_id"].astype(str).str.strip().str.zfill(2)

if "municipio_id" in df.columns:
    df["municipio_id"] = df["municipio_id"].astype(str).str.strip().str.zfill(3)

if {"entidad_id", "municipio_id"}.issubset(df.columns):
    df["geo_key"] = df["entidad_id"] + df["municipio_id"]
elif "geo_key" in df.columns:
    df["geo_key"] = df["geo_key"].astype(str).str.strip().str.zfill(5)


# ------------------------------------------------------------
# 8. Reporte de carga
# ------------------------------------------------------------

print("\nARCHIVO CARGADO CORRECTAMENTE")
print("=" * 100)
print(f"Ruta cargada: {selected_path}")
print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")

print("\nColumnas disponibles:")
print("=" * 100)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\nVista rápida:")
display(df.head())

Mounted at /content/drive
Project path:
/content/drive/MyDrive/Nearshoring_Project

Processed path:
/content/drive/MyDrive/Nearshoring_Project/data/processed

Modeling output path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base

Coincidencias encontradas:
0 -> [municipality_industrial_b2b_opportunity_panel_curated] /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_curated.csv
1 -> [municipality_industrial_b2b_opportunity_panel] /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel.csv
2 -> [municipality_industrial_b2b_opportunity_panel_raw] /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_raw.csv

Archivo seleccionado:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_curated.csv

ARCHIVO CARGADO CORRECTAMENTE
Ruta cargada: /content/drive/MyDrive/Nears

,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_capacity_score,industrial_capacity_percentile,rank_industrial_capacity_within_year,industrial_capacity_group,...,b2b_large_per_100_manufacturing_establishments,b2b_support_per_1000_manufacturing_workers,b2b_medium_large_per_1000_manufacturing_workers,b2b_large_per_1000_manufacturing_workers,high_industrial_capacity,high_b2b_support,high_b2b_medium_large,high_b2b_large,industrial_b2b_typology_broad,industrial_b2b_typology_scaled
0,2018,15,México,106,Toluca,15106,0.992818,1.000000,1.0,Top industrial hub,...,1.631809,28.756669,1.746453,0.849303,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.992049,0.999555,2.0,Top industrial hub,...,2.327406,24.115603,1.689829,0.702780,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
2,2018,02,Baja California,004,Tijuana,02004,0.990813,0.999109,3.0,Top industrial hub,...,2.179837,15.830109,0.881302,0.296236,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
3,2018,22,Querétaro,014,Querétaro,22014,0.989939,0.998664,4.0,Top industrial hub,...,3.033006,34.275266,2.505587,0.986756,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,0.989542,0.998218,5.0,Top industrial hub,...,3.861193,27.667200,2.379974,1.119155,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub


In [ ]:
# ============================================================
# 03. MUESTRA ALEATORIA: ID BÁSICO + FEATURES CANDIDATAS
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona desde cero después de cerrar sesión.
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
#
# Output:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/13_modeling_base_feature_sample.csv
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive de forma segura
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive ya está disponible.")


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

output_path = modeling_path / "13_modeling_base_feature_sample.csv"

print("Input path:")
print(input_path)

print("\nOutput path:")
print(output_path)

if not input_path.exists():
    raise FileNotFoundError(
        f"No encontré el archivo input en: {input_path}"
    )


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df_model = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df_model["year"] = pd.to_numeric(
    df_model["year"],
    errors="coerce"
).astype("Int64")

df_model["entidad_id"] = (
    df_model["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df_model["municipio_id"] = (
    df_model["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df_model["geo_key"] = (
    df_model["entidad_id"] + df_model["municipio_id"]
)

print("\nBase cargada:")
print(f"Filas: {df_model.shape[0]:,}")
print(f"Columnas: {df_model.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir columnas: SOLO ID básico + features candidatas
# ------------------------------------------------------------

id_cols = [
    "year",
    "entidad_name",
    "municipio_name",
    "geo_key"
]

feature_cols = [
    # Capacidad manufacturera
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",
    "denue_manufacturing_establishments",

    # Oferta B2B
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    # Tamaño B2B
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    # Shares B2B
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments",

    # Ratios B2B / manufactura
    "b2b_support_per_100_manufacturing_establishments",
    "b2b_medium_large_per_100_manufacturing_establishments",
    "b2b_large_per_100_manufacturing_establishments",
    "b2b_support_per_1000_manufacturing_workers",
    "b2b_medium_large_per_1000_manufacturing_workers",
    "b2b_large_per_1000_manufacturing_workers"
]


# ------------------------------------------------------------
# 4. Seleccionar solo columnas existentes
# ------------------------------------------------------------

selected_cols = id_cols + feature_cols

existing_cols = [
    col for col in selected_cols
    if col in df_model.columns
]

missing_cols = [
    col for col in selected_cols
    if col not in df_model.columns
]

df_selected = df_model[existing_cols].copy()

print("\nColumnas seleccionadas:")
print(f"Existentes: {len(existing_cols)}")
print(f"Faltantes: {len(missing_cols)}")

if missing_cols:
    print("\nColumnas faltantes:")
    for col in missing_cols:
        print("-", col)


# ------------------------------------------------------------
# 5. Crear muestra aleatoria pequeña
# ------------------------------------------------------------

sample_size = 50
random_seed = 123

sample_size = min(sample_size, len(df_selected))

df_sample = (
    df_selected
    .sample(
        n=sample_size,
        random_state=random_seed
    )
    .sort_values(["year", "entidad_name", "municipio_name"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Guardar muestra
# ------------------------------------------------------------

df_sample.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nMuestra creada correctamente")
print("=" * 100)
print(f"Filas: {df_sample.shape[0]:,}")
print(f"Columnas: {df_sample.shape[1]:,}")

print("\nArchivo guardado en:")
print(output_path)

display(df_sample)

Google Drive ya está disponible.
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Output path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/13_modeling_base_feature_sample.csv

Base cargada:
Filas: 4,530
Columnas: 54

Columnas seleccionadas:
Existentes: 31
Faltantes: 0

Muestra creada correctamente
Filas: 50
Columnas: 31

Archivo guardado en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/13_modeling_base_feature_sample.csv


,year,entidad_name,municipio_name,geo_key,total_manufacturing_establishments,total_manufacturing_employment,total_manufacturing_value_added,total_manufacturing_income,total_manufacturing_investment,denue_manufacturing_establishments,...,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments,b2b_support_per_100_manufacturing_establishments,b2b_medium_large_per_100_manufacturing_establishments,b2b_large_per_100_manufacturing_establishments,b2b_support_per_1000_manufacturing_workers,b2b_medium_large_per_1000_manufacturing_workers,b2b_large_per_1000_manufacturing_workers
0,2018,Hidalgo,Pisaflores,13049,30.0,49.0,2.346,6.446,0.365,30.0,...,0.000000,0.000000,0.000000,0.000000,23.333333,0.000000,0.000000,142.857143,0.000000,0.000000
1,2018,Jalisco,Acatlán de Juárez,14002,80.0,1227.0,909.253,2323.086,27.141,84.0,...,0.058824,0.000000,0.000000,0.000000,42.500000,0.000000,0.000000,27.709861,0.000000,0.000000
2,2018,Jalisco,Degollado,14033,169.0,683.0,67.492,169.123,0.462,164.0,...,0.000000,0.000000,0.000000,0.000000,27.218935,0.000000,0.000000,67.349927,0.000000,0.000000
3,2018,Jalisco,Juanacatlán,14051,33.0,628.0,45.653,496.985,20.842,44.0,...,0.071429,0.071429,0.000000,0.071429,42.424242,3.030303,0.000000,22.292994,1.592357,0.000000
4,2018,Michoacán de Ocampo,Pátzcuaro,16066,1022.0,2302.0,159.563,451.710,6.363,936.0,...,0.069444,0.004630,0.004630,0.009259,21.135029,0.195695,0.097847,93.831451,0.868810,0.434405
5,2018,México,Lerma,15051,1000.0,47458.0,30196.534,97585.051,2648.133,931.0,...,0.122302,0.053957,0.035971,0.089928,27.800000,2.500000,1.000000,5.857811,0.526782,0.210713
6,2018,Oaxaca,San Antonio Nanahuatípam,20109,5.0,8.0,0.495,0.845,0.030,3.0,...,0.000000,0.000000,0.000000,0.000000,20.000000,0.000000,0.000000,125.000000,0.000000,0.000000
7,2018,Oaxaca,San Juan Lachigalla,20203,21.0,58.0,0.709,1.497,0.260,2.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,2018,Oaxaca,San Pablo Coatlán,20291,11.0,17.0,1.310,2.333,0.000,7.0,...,0.000000,0.000000,0.000000,0.000000,18.181818,0.000000,0.000000,117.647059,0.000000,0.000000
9,2018,Puebla,Ajalpan,21010,2371.0,8059.0,486.778,890.787,21.721,1620.0,...,0.014286,0.014286,0.000000,0.014286,2.952341,0.042176,0.000000,8.685941,0.124085,0.000000


In [ ]:
# ============================================================
# 04. VARIABLE DE CONSISTENCIA ENTRE SAIC Y DENUE MANUFACTURA
# Nearshoring Project / Market Opportunity Analytics
#
# Este script funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Construir variables auxiliares de consistencia entre:
# - total_manufacturing_establishments
# - denue_manufacturing_establishments
#
# Estas variables NO serán input del scoring/clustering.
# Sirven para auditoría, interpretación y conteo de consistencia.
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
#
# Outputs:
# 14_modeling_base_with_manufacturing_consistency.csv
# 15_manufacturing_source_consistency_summary.csv
# 16_manufacturing_source_consistency_examples.csv
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive de forma segura
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive ya está disponible.")


# ------------------------------------------------------------
# 1. Definir rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

output_base_path = (
    modeling_path / "14_modeling_base_with_manufacturing_consistency.csv"
)

summary_output_path = (
    modeling_path / "15_manufacturing_source_consistency_summary.csv"
)

examples_output_path = (
    modeling_path / "16_manufacturing_source_consistency_examples.csv"
)

print("Input path:")
print(input_path)

print("\nOutput path:")
print(output_base_path)

if not input_path.exists():
    raise FileNotFoundError(
        f"No encontré el archivo input en: {input_path}"
    )


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
).astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = (
    df["entidad_id"] + df["municipio_id"]
)

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Validar columnas necesarias
# ------------------------------------------------------------

saic_col = "total_manufacturing_establishments"
denue_col = "denue_manufacturing_establishments"

required_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    saic_col,
    denue_col
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Faltan columnas necesarias en la base: {missing_cols}"
    )

df[saic_col] = pd.to_numeric(
    df[saic_col],
    errors="coerce"
)

df[denue_col] = pd.to_numeric(
    df[denue_col],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Calcular percentiles por año
# ------------------------------------------------------------
# La comparación se hace dentro de cada año.
# Así evitamos comparar niveles absolutos de 2018 contra 2023.

df["saic_manufacturing_establishments_percentile"] = (
    df
    .groupby("year")[saic_col]
    .rank(pct=True, method="average")
)

df["denue_manufacturing_establishments_percentile"] = (
    df
    .groupby("year")[denue_col]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 5. Calcular diferencia percentil y score de consistencia
# ------------------------------------------------------------

df["manufacturing_source_percentile_gap"] = (
    df["saic_manufacturing_establishments_percentile"]
    - df["denue_manufacturing_establishments_percentile"]
).abs()

df["manufacturing_source_consistency_score"] = (
    1 - df["manufacturing_source_percentile_gap"]
)


# ------------------------------------------------------------
# 6. Crear variable categórica de consistencia
# ------------------------------------------------------------
# Regla:
# gap <= 0.15        -> Alta consistencia
# 0.15 < gap <= 0.30 -> Consistencia media
# gap > 0.30         -> Baja consistencia

def classify_consistency(gap):
    if pd.isna(gap):
        return "Sin información"
    elif gap <= 0.15:
        return "Alta consistencia"
    elif gap <= 0.30:
        return "Consistencia media"
    else:
        return "Baja consistencia"


df["manufacturing_source_consistency_group"] = (
    df["manufacturing_source_percentile_gap"]
    .apply(classify_consistency)
)


# ------------------------------------------------------------
# 7. Resumen por año y grupo
# ------------------------------------------------------------

consistency_summary = (
    df
    .groupby(
        ["year", "manufacturing_source_consistency_group"],
        dropna=False
    )
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_gap=("manufacturing_source_percentile_gap", "mean"),
        avg_consistency_score=("manufacturing_source_consistency_score", "mean"),
        avg_saic_manufacturing_establishments=(saic_col, "mean"),
        avg_denue_manufacturing_establishments=(denue_col, "mean")
    )
    .reset_index()
)

consistency_summary["share_municipality_years"] = (
    consistency_summary["n_municipality_years"]
    / consistency_summary.groupby("year")["n_municipality_years"].transform("sum")
)

# Orden más legible
group_order = {
    "Alta consistencia": 1,
    "Consistencia media": 2,
    "Baja consistencia": 3,
    "Sin información": 4
}

consistency_summary["group_order"] = (
    consistency_summary["manufacturing_source_consistency_group"]
    .map(group_order)
)

consistency_summary = (
    consistency_summary
    .sort_values(["year", "group_order"])
    .drop(columns=["group_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Tabla de ejemplos para revisar municipios
# ------------------------------------------------------------

example_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    saic_col,
    denue_col,
    "saic_manufacturing_establishments_percentile",
    "denue_manufacturing_establishments_percentile",
    "manufacturing_source_percentile_gap",
    "manufacturing_source_consistency_score",
    "manufacturing_source_consistency_group"
]

consistency_examples = (
    df[example_cols]
    .sort_values(
        ["year", "manufacturing_source_percentile_gap"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Guardar outputs
# ------------------------------------------------------------

df.to_csv(
    output_base_path,
    index=False,
    encoding="utf-8-sig"
)

consistency_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig"
)

consistency_examples.to_csv(
    examples_output_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 10. Mostrar resultados
# ------------------------------------------------------------

print("\nRESUMEN DE CONSISTENCIA")
print("=" * 100)
display(consistency_summary)

print("\nEJEMPLOS CON MAYOR DIFERENCIA ENTRE FUENTES")
print("=" * 100)
display(consistency_examples.head(30))

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas originales:", len(df))
print("Missing en percentil SAIC:", df["saic_manufacturing_establishments_percentile"].isna().sum())
print("Missing en percentil DENUE:", df["denue_manufacturing_establishments_percentile"].isna().sum())
print("Missing en gap:", df["manufacturing_source_percentile_gap"].isna().sum())

print("\nArchivos guardados en:")
print(modeling_path)

print("\nArchivo principal con consistencia:")
print(output_base_path)

print("\nResumen de consistencia:")
print(summary_output_path)

print("\nEjemplos de consistencia:")
print(examples_output_path)

Mounted at /content/drive
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Output path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/14_modeling_base_with_manufacturing_consistency.csv

Base cargada:
Filas: 4,530
Columnas: 54

RESUMEN DE CONSISTENCIA


,year,manufacturing_source_consistency_group,n_municipality_years,n_municipalities,avg_gap,avg_consistency_score,avg_saic_manufacturing_establishments,avg_denue_manufacturing_establishments,share_municipality_years
0,2018,Alta consistencia,2173,2173,0.037140,0.962860,263.509434,242.213530,0.967929
1,2018,Consistencia media,61,61,0.190095,0.809905,76.754098,43.655738,0.027171
2,2018,Baja consistencia,11,11,0.428002,0.571998,148.909091,17.363636,0.004900
3,2023,Alta consistencia,2238,2238,0.034487,0.965513,281.162645,271.339589,0.979431
4,2023,Consistencia media,38,38,0.191812,0.808188,76.552632,43.763158,0.016630
5,2023,Baja consistencia,9,9,0.530975,0.469025,156.000000,105.333333,0.003939



EJEMPLOS CON MAYOR DIFERENCIA ENTRE FUENTES


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,total_manufacturing_establishments,denue_manufacturing_establishments,saic_manufacturing_establishments_percentile,denue_manufacturing_establishments_percentile,manufacturing_source_percentile_gap,manufacturing_source_consistency_score,manufacturing_source_consistency_group
0,2018,17,Morelos,035,Xoxocotla,17035,208.0,0.0,0.738976,0.001336,0.737639,0.262361,Baja consistencia
1,2018,20,Oaxaca,015,Coatecas Altas,20015,142.0,3.0,0.661693,0.019376,0.642316,0.357684,Baja consistencia
2,2018,20,Oaxaca,534,San Vicente Coatlán,20534,215.0,16.0,0.746993,0.197773,0.549220,0.450780,Baja consistencia
3,2018,17,Morelos,034,Coatetelco,17034,74.0,0.0,0.506904,0.001336,0.505568,0.494432,Baja consistencia
4,2018,20,Oaxaca,182,San Juan Bautista Tlacoatzintepec,20182,44.0,2.0,0.371492,0.010022,0.361470,0.638530,Baja consistencia
5,2018,20,Oaxaca,353,Santa Ana,20353,70.0,13.0,0.494209,0.161693,0.332517,0.667483,Baja consistencia
6,2018,20,Oaxaca,367,Santa Catarina Mechoacán,20367,140.0,30.0,0.657461,0.328508,0.328953,0.671047,Baja consistencia
7,2018,20,Oaxaca,300,San Pedro Amuzgos,20300,477.0,76.0,0.873497,0.551225,0.322272,0.677728,Baja consistencia
8,2018,31,Yucatán,017,Chankom,31017,47.0,7.0,0.389310,0.073942,0.315367,0.684633,Baja consistencia
9,2018,20,Oaxaca,447,Santa María Zacatepec,20447,184.0,41.0,0.715145,0.403118,0.312027,0.687973,Baja consistencia



VALIDACIÓN RÁPIDA
Filas originales: 4530
Missing en percentil SAIC: 0
Missing en percentil DENUE: 0
Missing en gap: 0

Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base

Archivo principal con consistencia:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/14_modeling_base_with_manufacturing_consistency.csv

Resumen de consistencia:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/15_manufacturing_source_consistency_summary.csv

Ejemplos de consistencia:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/16_manufacturing_source_consistency_examples.csv


In [ ]:
# ============================================================
# 05. DIAGNÓSTICO DEL INDUSTRIAL DEMAND SCORE
# Nearshoring Project / Market Opportunity Analytics
#
# Este script funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Calcular y revisar el componente de demanda industrial,
# sin guardar todavía una nueva base completa de scoring.
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
#
# Outputs:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/
#   01_industrial_demand_score_diagnostic.csv
#   02_industrial_demand_score_summary.csv
#   03_industrial_demand_top_municipalities.csv
#   04_industrial_demand_score_weights.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje seguro de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

drive_root = Path("/content/drive/MyDrive")

if not drive_root.exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive ya está disponible.")


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

diagnostics_path = modeling_path / "scoring_diagnostics"
diagnostics_path.mkdir(parents=True, exist_ok=True)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

demand_diagnostic_path = (
    diagnostics_path / "01_industrial_demand_score_diagnostic.csv"
)

demand_summary_path = (
    diagnostics_path / "02_industrial_demand_score_summary.csv"
)

top_demand_path = (
    diagnostics_path / "03_industrial_demand_top_municipalities.csv"
)

weights_path = (
    diagnostics_path / "04_industrial_demand_score_weights.csv"
)

print("Input path:")
print(input_path)

print("\nDiagnostics path:")
print(diagnostics_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
).astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir variables y pesos del score de demanda
# ------------------------------------------------------------

demand_features = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

demand_weights = {
    "total_manufacturing_establishments": 0.25,
    "total_manufacturing_employment": 0.20,
    "total_manufacturing_value_added": 0.25,
    "total_manufacturing_income": 0.20,
    "total_manufacturing_investment": 0.10
}

missing_features = [
    col for col in demand_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Faltan variables para demanda: {missing_features}")

if not np.isclose(sum(demand_weights.values()), 1.0):
    raise ValueError("Los pesos del score de demanda no suman 1.")


# ------------------------------------------------------------
# 4. Validar y convertir variables
# ------------------------------------------------------------

for col in demand_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_demand = df[demand_features].isna().sum()

print("\nMissing en variables de demanda:")
print(missing_demand)

if missing_demand.sum() > 0:
    raise ValueError(
        "Hay missing en variables de demanda. "
        "Revisa que estés usando 08_modeling_base_complete_cases.csv."
    )


# ------------------------------------------------------------
# 5. Normalización por percentiles dentro de cada año
# ------------------------------------------------------------

for col in demand_features:
    percentile_col = f"{col}_demand_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )


# ------------------------------------------------------------
# 6. Construir industrial_demand_score
# ------------------------------------------------------------

df["industrial_demand_score"] = 0.0

for col, weight in demand_weights.items():
    percentile_col = f"{col}_demand_percentile"
    df["industrial_demand_score"] += df[percentile_col] * weight

df["industrial_demand_percentile"] = (
    df
    .groupby("year")["industrial_demand_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 7. Grupo interpretable
# ------------------------------------------------------------

def classify_demand_group(p):
    if pd.isna(p):
        return "Sin información"
    elif p >= 0.90:
        return "Top industrial demand"
    elif p >= 0.75:
        return "High industrial demand"
    elif p >= 0.50:
        return "Medium industrial demand"
    else:
        return "Low industrial demand"


df["industrial_demand_group"] = (
    df["industrial_demand_percentile"]
    .apply(classify_demand_group)
)


# ------------------------------------------------------------
# 8. Crear tabla diagnóstica compacta
# ------------------------------------------------------------

diagnostic_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    *demand_features,
    *[f"{col}_demand_percentile" for col in demand_features],
    "industrial_demand_score",
    "industrial_demand_percentile",
    "industrial_demand_group"
]

demand_diagnostic = df[diagnostic_cols].copy()


# ------------------------------------------------------------
# 9. Resumen por año y grupo
# ------------------------------------------------------------

demand_summary = (
    demand_diagnostic
    .groupby(["year", "industrial_demand_group"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_industrial_demand_score=("industrial_demand_score", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_manufacturing_establishments=("total_manufacturing_establishments", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_manufacturing_value_added=("total_manufacturing_value_added", "mean"),
        avg_manufacturing_income=("total_manufacturing_income", "mean"),
        avg_manufacturing_investment=("total_manufacturing_investment", "mean")
    )
    .reset_index()
)

demand_summary["share_municipality_years"] = (
    demand_summary["n_municipality_years"]
    / demand_summary.groupby("year")["n_municipality_years"].transform("sum")
)

group_order = {
    "Top industrial demand": 1,
    "High industrial demand": 2,
    "Medium industrial demand": 3,
    "Low industrial demand": 4,
    "Sin información": 5
}

demand_summary["group_order"] = (
    demand_summary["industrial_demand_group"]
    .map(group_order)
)

demand_summary = (
    demand_summary
    .sort_values(["year", "group_order"])
    .drop(columns=["group_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Top municipios por año
# ------------------------------------------------------------

top_demand = (
    demand_diagnostic
    .sort_values(
        ["year", "industrial_demand_score"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Tabla de pesos
# ------------------------------------------------------------

weights_table = pd.DataFrame({
    "feature": list(demand_weights.keys()),
    "weight": list(demand_weights.values()),
    "score_block": "industrial_demand_score",
    "normalization": "Percentile rank within year",
    "interpretation": [
        "Base manufacturera amplia; aproxima número potencial de clientes",
        "Escala operativa de la manufactura local",
        "Intensidad productiva y generación de valor",
        "Escala económica del mercado manufacturero",
        "Dinamismo productivo; menor peso por volatilidad"
    ]
})


# ------------------------------------------------------------
# 12. Guardar diagnósticos
# ------------------------------------------------------------

demand_diagnostic.to_csv(
    demand_diagnostic_path,
    index=False,
    encoding="utf-8-sig"
)

demand_summary.to_csv(
    demand_summary_path,
    index=False,
    encoding="utf-8-sig"
)

top_demand.to_csv(
    top_demand_path,
    index=False,
    encoding="utf-8-sig"
)

weights_table.to_csv(
    weights_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 13. Mostrar resultados
# ------------------------------------------------------------

print("\nPESOS")
display(weights_table)

print("\nRESUMEN DEL SCORE DE DEMANDA")
display(demand_summary)

print("\nTOP MUNICIPIOS POR DEMANDA INDUSTRIAL")
display(top_demand.head(30))

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas diagnosticadas:", len(demand_diagnostic))
print("Missing industrial_demand_score:", demand_diagnostic["industrial_demand_score"].isna().sum())
print("Missing industrial_demand_percentile:", demand_diagnostic["industrial_demand_percentile"].isna().sum())
print("Años:", sorted(demand_diagnostic["year"].dropna().unique()))

print("\nArchivos guardados en:")
print(diagnostics_path)

Google Drive ya está disponible.
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Diagnostics path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics

Base cargada:
Filas: 4,530
Columnas: 54

Missing en variables de demanda:
total_manufacturing_establishments    0
total_manufacturing_employment        0
total_manufacturing_value_added       0
total_manufacturing_income            0
total_manufacturing_investment        0
dtype: int64

PESOS


,feature,weight,score_block,normalization,interpretation
0,total_manufacturing_establishments,0.25,industrial_demand_score,Percentile rank within year,Base manufacturera amplia; aproxima número pot...
1,total_manufacturing_employment,0.20,industrial_demand_score,Percentile rank within year,Escala operativa de la manufactura local
2,total_manufacturing_value_added,0.25,industrial_demand_score,Percentile rank within year,Intensidad productiva y generación de valor
3,total_manufacturing_income,0.20,industrial_demand_score,Percentile rank within year,Escala económica del mercado manufacturero
4,total_manufacturing_investment,0.10,industrial_demand_score,Percentile rank within year,Dinamismo productivo; menor peso por volatilidad



RESUMEN DEL SCORE DE DEMANDA


,year,industrial_demand_group,n_municipality_years,n_municipalities,avg_industrial_demand_score,avg_industrial_demand_percentile,avg_manufacturing_establishments,avg_manufacturing_employment,avg_manufacturing_value_added,avg_manufacturing_income,avg_manufacturing_investment,share_municipality_years
0,2018,Top industrial demand,225,225,0.932010,0.950111,1347.062222,24004.884444,12906.339018,44234.194956,868.796707,0.100223
1,2018,High industrial demand,337,337,0.792753,0.824944,408.050445,2073.697329,663.823421,2511.692582,62.575967,0.150111
2,2018,Medium industrial demand,561,561,0.614466,0.624944,173.322638,534.165775,104.028319,511.279567,0.915439,0.249889
3,2018,Low industrial demand,1122,1122,0.268649,0.250111,36.622103,77.178253,4.760471,15.059394,0.112602,0.499777
4,2023,Top industrial demand,229,229,0.927890,0.950109,1410.205240,26176.558952,20923.204515,55949.689664,971.931192,0.100219
5,2023,High industrial demand,343,343,0.788304,0.824945,454.285714,2250.559767,1497.668501,5881.128854,0.694915,0.150109
6,2023,Medium industrial demand,571,571,0.614636,0.624945,186.845884,562.936953,134.185007,353.465674,2.785671,0.249891
7,2023,Low industrial demand,1142,1142,0.270725,0.250109,42.126970,88.134851,7.700065,18.856389,0.141715,0.499781



TOP MUNICIPIOS POR DEMANDA INDUSTRIAL


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,total_manufacturing_establishments,total_manufacturing_employment,total_manufacturing_value_added,total_manufacturing_income,total_manufacturing_investment,total_manufacturing_establishments_demand_percentile,total_manufacturing_employment_demand_percentile,total_manufacturing_value_added_demand_percentile,total_manufacturing_income_demand_percentile,total_manufacturing_investment_demand_percentile,industrial_demand_score,industrial_demand_percentile,industrial_demand_group
0,2018,15,México,106,Toluca,15106,4351.0,83598.0,97605.998,334675.563,3917.262,0.995991,0.994655,1.000000,1.000000,0.993764,0.997305,1.000000,Top industrial demand
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,3824.0,126640.0,77223.073,274960.549,8373.471,0.993318,0.998218,0.998218,0.998218,0.999555,0.997127,0.999555,Top industrial demand
2,2018,02,Baja California,004,Tijuana,02004,3670.0,270055.0,85526.588,222120.705,5618.985,0.992428,0.999555,0.999109,0.995991,0.998218,0.996815,0.999109,Top industrial demand
3,2018,14,Jalisco,039,Guadalajara,14039,8007.0,113447.0,51727.982,163458.345,2409.978,0.999555,0.996882,0.994655,0.993318,0.987528,0.995345,0.998664,Top industrial demand
4,2018,22,Querétaro,014,Querétaro,22014,3363.0,103369.0,60661.916,229532.904,12126.712,0.991091,0.995991,0.995100,0.996882,1.000000,0.995122,0.998218,Top industrial demand
5,2018,01,Aguascalientes,001,Aguascalientes,01001,3896.0,71120.0,71526.587,257820.921,3845.502,0.994209,0.991982,0.996882,0.997773,0.993318,0.995056,0.997773,Top industrial demand
6,2018,14,Jalisco,120,Zapopan,14120,4395.0,121731.0,51124.248,142709.014,2894.772,0.996437,0.997327,0.993764,0.990646,0.989310,0.994076,0.997327,Top industrial demand
7,2018,19,Nuevo León,039,Monterrey,19039,4729.0,75640.0,49357.178,151078.236,3248.158,0.997327,0.994209,0.993318,0.991091,0.992428,0.993964,0.996882,Top industrial demand
8,2018,08,Chihuahua,037,Juárez,08037,2375.0,329725.0,92715.360,157045.359,3176.689,0.985301,1.000000,0.999555,0.992428,0.991537,0.993853,0.996437,Top industrial demand
9,2018,05,Coahuila de Zaragoza,030,Saltillo,05030,2654.0,71941.0,74250.522,296390.165,3237.461,0.987528,0.992428,0.997773,0.999109,0.991982,0.993831,0.995991,Top industrial demand



VALIDACIÓN RÁPIDA
Filas diagnosticadas: 4530
Missing industrial_demand_score: 0
Missing industrial_demand_percentile: 0
Años: [np.int64(2018), np.int64(2023)]

Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics


In [ ]:
# ============================================================
# 06. DIAGNÓSTICO DEL B2B SUPPLY SCORE
# Nearshoring Project / Market Opportunity Analytics
#
# Este script funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Calcular y revisar el componente de oferta B2B observable,
# sin guardar todavía una nueva base completa de scoring.
#
# Lógica acordada:
# - No usar denue_b2b_support_establishments dentro del score
#   para evitar doble conteo con sus componentes.
#
# B2B supply score =
# 0.35 * P(logística)
# + 0.25 * P(servicios profesionales/técnicos)
# + 0.20 * P(apoyo a negocios)
# + 0.20 * P(diversidad SCIAN B2B)
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
#
# Outputs:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/
#   05_b2b_supply_score_diagnostic.csv
#   06_b2b_supply_score_summary.csv
#   07_b2b_supply_top_municipalities.csv
#   08_b2b_supply_score_weights.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import os


def get_drive_root():
    """
    Devuelve la ruta de MyDrive si ya está montado.
    Si no está montado, intenta montarlo.
    Si falla, intenta force_remount.
    """

    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    # 1. Si Drive ya está montado, usarlo
    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    # 2. Intentar montar en /content/drive
    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    # 3. Intentar force_remount
    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    # 4. Si nada funcionó
    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Solución práctica: en Colab ve a Runtime > Restart runtime, "
        "vuelve a ejecutar la celda y autoriza Drive otra vez."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

diagnostics_path = modeling_path / "scoring_diagnostics"
diagnostics_path.mkdir(parents=True, exist_ok=True)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

b2b_diagnostic_path = (
    diagnostics_path / "05_b2b_supply_score_diagnostic.csv"
)

b2b_summary_path = (
    diagnostics_path / "06_b2b_supply_score_summary.csv"
)

top_b2b_path = (
    diagnostics_path / "07_b2b_supply_top_municipalities.csv"
)

weights_path = (
    diagnostics_path / "08_b2b_supply_score_weights.csv"
)

print("\nProject path:")
print(project_path)

print("\nInput path:")
print(input_path)

print("\nDiagnostics path:")
print(diagnostics_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
).astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Variables y pesos del B2B supply score
# ------------------------------------------------------------
# Nota metodológica:
# No usamos denue_b2b_support_establishments dentro del score
# para evitar doble conteo con sus componentes.
#
# Pesos:
# logística: 0.35
# servicios profesionales/técnicos: 0.25
# apoyo a negocios: 0.20
# diversidad SCIAN B2B: 0.20

b2b_supply_features = [
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes"
]

b2b_supply_weights = {
    "denue_logistics_storage_establishments": 0.35,
    "denue_professional_technical_establishments": 0.25,
    "denue_business_support_establishments": 0.20,
    "denue_b2b_support_scian_classes": 0.20
}

missing_features = [
    col for col in b2b_supply_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(f"Faltan variables para oferta B2B: {missing_features}")

if not np.isclose(sum(b2b_supply_weights.values()), 1.0):
    raise ValueError("Los pesos del B2B supply score no suman 1.")


# ------------------------------------------------------------
# 4. Validar y convertir variables
# ------------------------------------------------------------

for col in b2b_supply_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_b2b = df[b2b_supply_features].isna().sum()

print("\nMissing en variables de oferta B2B:")
print(missing_b2b)

if missing_b2b.sum() > 0:
    raise ValueError(
        "Hay missing en variables B2B. "
        "Revisa que estés usando 08_modeling_base_complete_cases.csv."
    )


# ------------------------------------------------------------
# 5. Normalización por percentiles dentro de cada año
# ------------------------------------------------------------

for col in b2b_supply_features:
    percentile_col = f"{col}_b2b_supply_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )


# ------------------------------------------------------------
# 6. Construir b2b_supply_score
# ------------------------------------------------------------

df["b2b_supply_score"] = 0.0

for col, weight in b2b_supply_weights.items():
    percentile_col = f"{col}_b2b_supply_percentile"
    df["b2b_supply_score"] += df[percentile_col] * weight

df["b2b_supply_percentile"] = (
    df
    .groupby("year")["b2b_supply_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 7. Grupo interpretable
# ------------------------------------------------------------

def classify_b2b_supply_group(p):
    if pd.isna(p):
        return "Sin información"
    elif p >= 0.90:
        return "Top B2B supply"
    elif p >= 0.75:
        return "High B2B supply"
    elif p >= 0.50:
        return "Medium B2B supply"
    else:
        return "Low B2B supply"


df["b2b_supply_group"] = (
    df["b2b_supply_percentile"]
    .apply(classify_b2b_supply_group)
)


# ------------------------------------------------------------
# 8. Tabla diagnóstica compacta
# ------------------------------------------------------------

diagnostic_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    # Variable descriptiva, NO usada en el score
    "denue_b2b_support_establishments",

    # Variables usadas en el score
    *b2b_supply_features,

    # Percentiles
    *[f"{col}_b2b_supply_percentile" for col in b2b_supply_features],

    # Score
    "b2b_supply_score",
    "b2b_supply_percentile",
    "b2b_supply_group"
]

diagnostic_cols = [
    col for col in diagnostic_cols
    if col in df.columns
]

b2b_diagnostic = df[diagnostic_cols].copy()


# ------------------------------------------------------------
# 9. Resumen por año y grupo
# ------------------------------------------------------------

b2b_summary = (
    b2b_diagnostic
    .groupby(["year", "b2b_supply_group"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_b2b_supply_score=("b2b_supply_score", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_b2b_total_descriptive=("denue_b2b_support_establishments", "mean"),
        avg_logistics_storage=("denue_logistics_storage_establishments", "mean"),
        avg_professional_technical=("denue_professional_technical_establishments", "mean"),
        avg_business_support=("denue_business_support_establishments", "mean"),
        avg_b2b_scian_classes=("denue_b2b_support_scian_classes", "mean")
    )
    .reset_index()
)

b2b_summary["share_municipality_years"] = (
    b2b_summary["n_municipality_years"]
    / b2b_summary.groupby("year")["n_municipality_years"].transform("sum")
)

group_order = {
    "Top B2B supply": 1,
    "High B2B supply": 2,
    "Medium B2B supply": 3,
    "Low B2B supply": 4,
    "Sin información": 5
}

b2b_summary["group_order"] = (
    b2b_summary["b2b_supply_group"]
    .map(group_order)
)

b2b_summary = (
    b2b_summary
    .sort_values(["year", "group_order"])
    .drop(columns=["group_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Top municipios por año
# ------------------------------------------------------------

top_b2b = (
    b2b_diagnostic
    .sort_values(
        ["year", "b2b_supply_score"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Tabla de pesos
# ------------------------------------------------------------

weights_table = pd.DataFrame({
    "feature": list(b2b_supply_weights.keys()),
    "weight": list(b2b_supply_weights.values()),
    "score_block": "b2b_supply_score",
    "normalization": "Percentile rank within year",
    "interpretation": [
        "Oferta logística y almacenamiento; mayor peso por relevancia en nearshoring",
        "Oferta profesional, técnica y especializada",
        "Oferta operativa y de apoyo empresarial",
        "Diversidad de clases SCIAN B2B presentes"
    ]
})


# ------------------------------------------------------------
# 12. Guardar diagnósticos
# ------------------------------------------------------------

b2b_diagnostic.to_csv(
    b2b_diagnostic_path,
    index=False,
    encoding="utf-8-sig"
)

b2b_summary.to_csv(
    b2b_summary_path,
    index=False,
    encoding="utf-8-sig"
)

top_b2b.to_csv(
    top_b2b_path,
    index=False,
    encoding="utf-8-sig"
)

weights_table.to_csv(
    weights_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 13. Mostrar resultados
# ------------------------------------------------------------

print("\nPESOS DEL B2B SUPPLY SCORE")
print("=" * 100)
display(weights_table)

print("\nRESUMEN DEL B2B SUPPLY SCORE")
print("=" * 100)
display(b2b_summary)

print("\nTOP MUNICIPIOS POR OFERTA B2B")
print("=" * 100)
display(top_b2b.head(30))

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas diagnosticadas:", len(b2b_diagnostic))
print("Missing b2b_supply_score:", b2b_diagnostic["b2b_supply_score"].isna().sum())
print("Missing b2b_supply_percentile:", b2b_diagnostic["b2b_supply_percentile"].isna().sum())
print("Años:", sorted(b2b_diagnostic["year"].dropna().unique()))

print("\nArchivos guardados en:")
print(diagnostics_path)

print("\nArchivos generados:")
print(b2b_diagnostic_path)
print(b2b_summary_path)
print(top_b2b_path)
print(weights_path)

Google Drive ya está disponible en: /content/drive/MyDrive

Project path:
/content/drive/MyDrive/Nearshoring_Project

Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Diagnostics path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics

Base cargada:
Filas: 4,530
Columnas: 54

Missing en variables de oferta B2B:
denue_logistics_storage_establishments         0
denue_professional_technical_establishments    0
denue_business_support_establishments          0
denue_b2b_support_scian_classes                0
dtype: int64

PESOS DEL B2B SUPPLY SCORE


,feature,weight,score_block,normalization,interpretation
0,denue_logistics_storage_establishments,0.35,b2b_supply_score,Percentile rank within year,Oferta logística y almacenamiento; mayor peso ...
1,denue_professional_technical_establishments,0.25,b2b_supply_score,Percentile rank within year,"Oferta profesional, técnica y especializada"
2,denue_business_support_establishments,0.20,b2b_supply_score,Percentile rank within year,Oferta operativa y de apoyo empresarial
3,denue_b2b_support_scian_classes,0.20,b2b_supply_score,Percentile rank within year,Diversidad de clases SCIAN B2B presentes



RESUMEN DEL B2B SUPPLY SCORE


,year,b2b_supply_group,n_municipality_years,n_municipalities,avg_b2b_supply_score,avg_b2b_supply_percentile,avg_b2b_total_descriptive,avg_logistics_storage,avg_professional_technical,avg_business_support,avg_b2b_scian_classes,share_municipality_years
0,2018,Top B2B supply,225,225,0.946121,0.950111,872.311111,132.333333,391.475556,348.502222,58.191111,0.100223
1,2018,High B2B supply,337,337,0.813972,0.824944,92.302671,15.350148,31.314540,45.637982,21.881306,0.150111
2,2018,Medium B2B supply,561,561,0.608399,0.624944,26.572193,4.611408,7.108734,14.852050,9.575758,0.249889
3,2018,Low B2B supply,1122,1122,0.262480,0.250111,5.160428,0.687166,0.834225,3.639037,2.559715,0.499777
4,2023,Top B2B supply,229,229,0.946545,0.950109,789.187773,139.393013,392.633188,257.161572,61.449782,0.100219
5,2023,High B2B supply,343,343,0.813241,0.824945,83.772595,15.381924,35.396501,32.994169,23.061224,0.150109
6,2023,Medium B2B supply,571,571,0.607477,0.624945,24.173380,4.231173,8.500876,11.441331,10.099825,0.249891
7,2023,Low B2B supply,1142,1142,0.263074,0.250109,4.551664,0.614711,1.074431,2.862522,2.500000,0.499781



TOP MUNICIPIOS POR OFERTA B2B


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,denue_logistics_storage_establishments_b2b_supply_percentile,denue_professional_technical_establishments_b2b_supply_percentile,denue_business_support_establishments_b2b_supply_percentile,denue_b2b_support_scian_classes_b2b_supply_percentile,b2b_supply_score,b2b_supply_percentile,b2b_supply_group
0,2018,14,Jalisco,039,Guadalajara,14039,7008.0,817.0,3751.0,2440.0,110.0,0.999555,0.999555,1.000000,1.000000,0.999733,1.000000,Top B2B supply
1,2018,02,Baja California,004,Tijuana,02004,4275.0,782.0,1905.0,1588.0,105.0,0.999109,0.997327,0.997773,0.998886,0.998352,0.999555,Top B2B supply
2,2018,21,Puebla,114,Puebla,21114,5014.0,611.0,2303.0,2100.0,107.0,0.997327,0.997773,0.999555,0.999555,0.998330,0.999109,Top B2B supply
3,2018,19,Nuevo León,039,Monterrey,19039,4737.0,627.0,2543.0,1567.0,102.0,0.998218,0.998664,0.997327,0.997327,0.997973,0.998664,Top B2B supply
4,2018,09,Ciudad de México,015,Cuauhtémoc,09015,6122.0,459.0,4022.0,1641.0,99.0,0.995100,1.000000,0.998218,0.995546,0.997038,0.998218,Top B2B supply
5,2018,11,Guanajuato,020,León,11020,4220.0,539.0,1837.0,1844.0,98.0,0.995991,0.996882,0.999109,0.994432,0.996526,0.997773,Top B2B supply
6,2018,22,Querétaro,014,Querétaro,22014,3543.0,427.0,1809.0,1307.0,103.0,0.994655,0.996437,0.996437,0.997773,0.996080,0.997327,Top B2B supply
7,2018,24,San Luis Potosí,028,San Luis Potosí,24028,3054.0,486.0,1448.0,1120.0,99.0,0.995546,0.995546,0.994209,0.995546,0.995278,0.996882,Top B2B supply
8,2018,09,Ciudad de México,016,Miguel Hidalgo,09016,3945.0,353.0,2510.0,1082.0,104.0,0.992428,0.998218,0.993318,0.998218,0.995212,0.996437,Top B2B supply
9,2018,31,Yucatán,050,Mérida,31050,3121.0,412.0,1575.0,1134.0,100.0,0.994209,0.995991,0.994655,0.996437,0.995189,0.995991,Top B2B supply



VALIDACIÓN RÁPIDA
Filas diagnosticadas: 4530
Missing b2b_supply_score: 0
Missing b2b_supply_percentile: 0
Años: [np.int64(2018), np.int64(2023)]

Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics

Archivos generados:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/05_b2b_supply_score_diagnostic.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/06_b2b_supply_score_summary.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/07_b2b_supply_top_municipalities.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/08_b2b_supply_score_weights.csv


In [ ]:
# ============================================================
# 06B. VISTAS INTERESANTES: DEMANDA INDUSTRIAL × OFERTA B2B
# Nearshoring Project / Market Opportunity Analytics
#
# Este script funciona aunque hayas cerrado sesión.
#
# Inputs:
# scoring_diagnostics/01_industrial_demand_score_diagnostic.csv
# scoring_diagnostics/05_b2b_supply_score_diagnostic.csv
#
# Outputs:
# scoring_diagnostics/09_top_b2b_supply_municipalities.csv
# scoring_diagnostics/10_bottom_b2b_supply_municipalities.csv
# scoring_diagnostics/11_high_demand_low_b2b_supply_candidates.csv
# scoring_diagnostics/12_high_demand_high_b2b_supply_hubs.csv
# scoring_diagnostics/13_demand_supply_cross_tab.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

diagnostics_path = (
    project_path / "data" / "processed" / "modeling_base" / "scoring_diagnostics"
)

demand_path = diagnostics_path / "01_industrial_demand_score_diagnostic.csv"
b2b_path = diagnostics_path / "05_b2b_supply_score_diagnostic.csv"

print("Demand input:")
print(demand_path)

print("\nB2B input:")
print(b2b_path)

if not demand_path.exists():
    raise FileNotFoundError(f"No encontré el diagnóstico de demanda: {demand_path}")

if not b2b_path.exists():
    raise FileNotFoundError(f"No encontré el diagnóstico de oferta B2B: {b2b_path}")


# ------------------------------------------------------------
# 2. Cargar diagnósticos
# ------------------------------------------------------------

demand = pd.read_csv(
    demand_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

b2b = pd.read_csv(
    b2b_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

for df in [demand, b2b]:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["entidad_id"] = df["entidad_id"].astype(str).str.strip().str.zfill(2)
    df["municipio_id"] = df["municipio_id"].astype(str).str.strip().str.zfill(3)
    df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nDemanda:")
print(demand.shape)

print("\nOferta B2B:")
print(b2b.shape)


# ------------------------------------------------------------
# 3. Seleccionar columnas y cruzar
# ------------------------------------------------------------

demand_keep = [
    "year",
    "geo_key",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "industrial_demand_score",
    "industrial_demand_percentile",
    "industrial_demand_group",
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

b2b_keep = [
    "year",
    "geo_key",
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",
    "b2b_supply_score",
    "b2b_supply_percentile",
    "b2b_supply_group"
]

demand_keep = [col for col in demand_keep if col in demand.columns]
b2b_keep = [col for col in b2b_keep if col in b2b.columns]

combined = demand[demand_keep].merge(
    b2b[b2b_keep],
    on=["year", "geo_key"],
    how="inner"
)

print("\nBase combinada:")
print(combined.shape)


# ------------------------------------------------------------
# 4. Crear etiquetas cruzadas
# ------------------------------------------------------------

def classify_cross(row):
    demand_p = row["industrial_demand_percentile"]
    b2b_p = row["b2b_supply_percentile"]

    if demand_p >= 0.75 and b2b_p < 0.50:
        return "High demand + low B2B supply"
    elif demand_p >= 0.75 and b2b_p >= 0.75:
        return "High demand + high B2B supply"
    elif demand_p < 0.50 and b2b_p >= 0.75:
        return "Low demand + high B2B supply"
    elif demand_p < 0.50 and b2b_p < 0.50:
        return "Low demand + low B2B supply"
    else:
        return "Middle cases"


combined["demand_supply_profile"] = combined.apply(classify_cross, axis=1)


# ------------------------------------------------------------
# 5. Tablas interesantes
# ------------------------------------------------------------

top_b2b_supply = (
    combined
    .sort_values(["year", "b2b_supply_score"], ascending=[True, False])
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

bottom_b2b_supply = (
    combined
    .sort_values(["year", "b2b_supply_score"], ascending=[True, True])
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

high_demand_low_b2b = (
    combined[
        (combined["industrial_demand_percentile"] >= 0.75) &
        (combined["b2b_supply_percentile"] < 0.50)
    ]
    .sort_values(
        ["year", "industrial_demand_percentile", "b2b_supply_percentile"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

high_demand_high_b2b = (
    combined[
        (combined["industrial_demand_percentile"] >= 0.75) &
        (combined["b2b_supply_percentile"] >= 0.75)
    ]
    .sort_values(
        ["year", "industrial_demand_percentile", "b2b_supply_percentile"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

cross_tab = (
    combined
    .groupby(["year", "demand_supply_profile"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_industrial_demand_score=("industrial_demand_score", "mean"),
        avg_b2b_supply_score=("b2b_supply_score", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_logistics=("denue_logistics_storage_establishments", "mean")
    )
    .reset_index()
)

cross_tab["share_municipality_years"] = (
    cross_tab["n_municipality_years"] /
    cross_tab.groupby("year")["n_municipality_years"].transform("sum")
)


# ------------------------------------------------------------
# 6. Guardar outputs
# ------------------------------------------------------------

top_b2b_supply.to_csv(
    diagnostics_path / "09_top_b2b_supply_municipalities.csv",
    index=False,
    encoding="utf-8-sig"
)

bottom_b2b_supply.to_csv(
    diagnostics_path / "10_bottom_b2b_supply_municipalities.csv",
    index=False,
    encoding="utf-8-sig"
)

high_demand_low_b2b.to_csv(
    diagnostics_path / "11_high_demand_low_b2b_supply_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

high_demand_high_b2b.to_csv(
    diagnostics_path / "12_high_demand_high_b2b_supply_hubs.csv",
    index=False,
    encoding="utf-8-sig"
)

cross_tab.to_csv(
    diagnostics_path / "13_demand_supply_cross_tab.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 7. Mostrar resultados
# ------------------------------------------------------------

print("\nTOP B2B SUPPLY")
display(top_b2b_supply.head(20))

print("\nBOTTOM B2B SUPPLY")
display(bottom_b2b_supply.head(20))

print("\nHIGH DEMAND + LOW B2B SUPPLY")
display(high_demand_low_b2b.head(30))

print("\nHIGH DEMAND + HIGH B2B SUPPLY")
display(high_demand_high_b2b.head(30))

print("\nCROSS TAB")
display(cross_tab)

print("\nArchivos guardados en:")
print(diagnostics_path)

Google Drive ya está disponible en: /content/drive/MyDrive
Demand input:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/01_industrial_demand_score_diagnostic.csv

B2B input:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/05_b2b_supply_score_diagnostic.csv

Demanda:
(4530, 19)

Oferta B2B:
(4530, 18)

Base combinada:
(4530, 22)

TOP B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,total_manufacturing_investment,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile
0,2018,14039,14,Jalisco,039,Guadalajara,0.995345,0.998664,Top industrial demand,8007.0,...,2409.978,7008.0,817.0,3751.0,2440.0,110.0,0.999733,1.000000,Top B2B supply,High demand + high B2B supply
1,2018,02004,02,Baja California,004,Tijuana,0.996815,0.999109,Top industrial demand,3670.0,...,5618.985,4275.0,782.0,1905.0,1588.0,105.0,0.998352,0.999555,Top B2B supply,High demand + high B2B supply
2,2018,21114,21,Puebla,114,Puebla,0.989087,0.993318,Top industrial demand,6849.0,...,1999.195,5014.0,611.0,2303.0,2100.0,107.0,0.998330,0.999109,Top B2B supply,High demand + high B2B supply
3,2018,19039,19,Nuevo León,039,Monterrey,0.993964,0.996882,Top industrial demand,4729.0,...,3248.158,4737.0,627.0,2543.0,1567.0,102.0,0.997973,0.998664,Top B2B supply,High demand + high B2B supply
4,2018,09015,09,Ciudad de México,015,Cuauhtémoc,0.976570,0.985746,Top industrial demand,3979.0,...,288.735,6122.0,459.0,4022.0,1641.0,99.0,0.997038,0.998218,Top B2B supply,High demand + high B2B supply
5,2018,11020,11,Guanajuato,020,León,0.992138,0.995546,Top industrial demand,10899.0,...,2270.654,4220.0,539.0,1837.0,1844.0,98.0,0.996526,0.997773,Top B2B supply,High demand + high B2B supply
6,2018,22014,22,Querétaro,014,Querétaro,0.995122,0.998218,Top industrial demand,3363.0,...,12126.712,3543.0,427.0,1809.0,1307.0,103.0,0.996080,0.997327,Top B2B supply,High demand + high B2B supply
7,2018,24028,24,San Luis Potosí,028,San Luis Potosí,0.997127,0.999555,Top industrial demand,3824.0,...,8373.471,3054.0,486.0,1448.0,1120.0,99.0,0.995278,0.996882,Top B2B supply,High demand + high B2B supply
8,2018,09016,09,Ciudad de México,016,Miguel Hidalgo,0.972294,0.979955,Top industrial demand,1233.0,...,1207.735,3945.0,353.0,2510.0,1082.0,104.0,0.995212,0.996437,Top B2B supply,High demand + high B2B supply
9,2018,31050,31,Yucatán,050,Mérida,0.976548,0.985301,Top industrial demand,3850.0,...,348.294,3121.0,412.0,1575.0,1134.0,100.0,0.995189,0.995991,Top B2B supply,High demand + high B2B supply



BOTTOM B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,total_manufacturing_investment,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile
0,2018,19016,19,Nuevo León,016,Doctor González,0.558018,0.559911,Medium industrial demand,8.0,...,54.833,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
1,2018,17035,17,Morelos,035,Xoxocotla,0.560624,0.563920,Medium industrial demand,208.0,...,-0.014,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
2,2018,31016,31,Yucatán,016,Chacsinkín,0.576325,0.584855,Medium industrial demand,448.0,...,-0.005,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
3,2018,17034,17,Morelos,034,Coatetelco,0.475234,0.465479,Low industrial demand,74.0,...,-0.009,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply
4,2018,20497,20,Oaxaca,497,Santiago Yaitepec,0.569644,0.575501,Medium industrial demand,278.0,...,0.071,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
5,2018,29003,29,Tlaxcala,003,Atlangatepec,0.491726,0.485078,Low industrial demand,7.0,...,10.594,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply
6,2018,20328,20,Oaxaca,328,San Pedro Taviche,0.555223,0.556347,Medium industrial demand,296.0,...,0.001,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
7,2018,20313,20,Oaxaca,313,San Pedro Jocotipac,0.520991,0.522049,Medium industrial demand,195.0,...,0.002,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases
8,2018,20533,20,Oaxaca,533,Santo Tomás Tamazulapan,0.425490,0.418263,Low industrial demand,15.0,...,0.571,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply
9,2018,20564,20,Oaxaca,564,Yutanduchi de Guerrero,0.515212,0.513586,Medium industrial demand,264.0,...,0.004,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases



HIGH DEMAND + LOW B2B SUPPLY


...


HIGH DEMAND + HIGH B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,total_manufacturing_investment,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile
0,2018,15106,15,México,106,Toluca,0.997305,1.000000,Top industrial demand,4351.0,...,3917.262,2404.0,311.0,885.0,1208.0,92.0,0.990802,0.992428,Top B2B supply,High demand + high B2B supply
1,2018,24028,24,San Luis Potosí,028,San Luis Potosí,0.997127,0.999555,Top industrial demand,3824.0,...,8373.471,3054.0,486.0,1448.0,1120.0,99.0,0.995278,0.996882,Top B2B supply,High demand + high B2B supply
2,2018,02004,02,Baja California,004,Tijuana,0.996815,0.999109,Top industrial demand,3670.0,...,5618.985,4275.0,782.0,1905.0,1588.0,105.0,0.998352,0.999555,Top B2B supply,High demand + high B2B supply
3,2018,14039,14,Jalisco,039,Guadalajara,0.995345,0.998664,Top industrial demand,8007.0,...,2409.978,7008.0,817.0,3751.0,2440.0,110.0,0.999733,1.000000,Top B2B supply,High demand + high B2B supply
4,2018,22014,22,Querétaro,014,Querétaro,0.995122,0.998218,Top industrial demand,3363.0,...,12126.712,3543.0,427.0,1809.0,1307.0,103.0,0.996080,0.997327,Top B2B supply,High demand + high B2B supply
5,2018,01001,01,Aguascalientes,001,Aguascalientes,0.995056,0.997773,Top industrial demand,3896.0,...,3845.502,2938.0,352.0,1395.0,1191.0,99.0,0.994098,0.995546,Top B2B supply,High demand + high B2B supply
6,2018,14120,14,Jalisco,120,Zapopan,0.994076,0.997327,Top industrial demand,4395.0,...,2894.772,2728.0,334.0,1177.0,1217.0,101.0,0.993898,0.995100,Top B2B supply,High demand + high B2B supply
7,2018,19039,19,Nuevo León,039,Monterrey,0.993964,0.996882,Top industrial demand,4729.0,...,3248.158,4737.0,627.0,2543.0,1567.0,102.0,0.997973,0.998664,Top B2B supply,High demand + high B2B supply
8,2018,08037,08,Chihuahua,037,Juárez,0.993853,0.996437,Top industrial demand,2375.0,...,3176.689,2135.0,553.0,799.0,783.0,93.0,0.992160,0.993764,Top B2B supply,High demand + high B2B supply
9,2018,05030,05,Coahuila de Zaragoza,030,Saltillo,0.993831,0.995991,Top industrial demand,2654.0,...,3237.461,1608.0,240.0,712.0,656.0,86.0,0.985089,0.987528,Top B2B supply,High demand + high B2B supply



CROSS TAB


,year,demand_supply_profile,n_municipality_years,n_municipalities,avg_industrial_demand_score,avg_b2b_supply_score,avg_manufacturing_employment,avg_b2b_total,avg_logistics,share_municipality_years
0,2018,High demand + high B2B supply,429,429,0.869226,0.888273,13580.337995,506.440559,76.895105,0.191091
1,2018,High demand + low B2B supply,23,23,0.777090,0.380611,2040.826087,13.956522,0.695652,0.010245
2,2018,Low demand + high B2B supply,5,5,0.447517,0.761236,138.200000,49.600000,12.400000,0.002227
3,2018,Low demand + low B2B supply,944,944,0.242500,0.249937,67.639831,4.417373,0.658898,0.420490
4,2018,Middle cases,844,844,0.593686,0.584633,650.176540,30.889810,5.471564,0.375947
5,2023,High demand + high B2B supply,436,436,0.864035,0.889232,14813.130734,459.068807,81.133028,0.190810
6,2023,High demand + low B2B supply,30,30,0.771449,0.381920,2319.266667,11.833333,0.533333,0.013129
7,2023,Low demand + high B2B supply,6,6,0.407566,0.763120,144.166667,45.500000,12.833333,0.002626
8,2023,Low demand + low B2B supply,963,963,0.245205,0.249678,77.427830,3.922118,0.595016,0.421444
9,2023,Middle cases,850,850,0.593599,0.586845,688.152941,28.117647,5.029412,0.371991



Archivos guardados en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics


In [ ]:
# ============================================================
# 06B. VISTAS DE DEMANDA INDUSTRIAL × OFERTA B2B
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Revisar en pantalla:
# 1. Top B2B supply
# 2. Bottom B2B supply
# 3. High demand + low B2B supply
# 4. High demand + high B2B supply
# 5. Tabla cruzada demanda-oferta
#
# NO GUARDA ARCHIVOS.
#
# Inputs:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/
#   01_industrial_demand_score_diagnostic.csv
#   05_b2b_supply_score_diagnostic.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    """
    Devuelve MyDrive si ya está montado.
    Si no, intenta montar Drive.
    """

    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

diagnostics_path = (
    project_path / "data" / "processed" / "modeling_base" / "scoring_diagnostics"
)

demand_path = diagnostics_path / "01_industrial_demand_score_diagnostic.csv"
b2b_path = diagnostics_path / "05_b2b_supply_score_diagnostic.csv"

print("Demand input:")
print(demand_path)

print("\nB2B input:")
print(b2b_path)

if not demand_path.exists():
    raise FileNotFoundError(f"No encontré el diagnóstico de demanda: {demand_path}")

if not b2b_path.exists():
    raise FileNotFoundError(f"No encontré el diagnóstico de oferta B2B: {b2b_path}")


# ------------------------------------------------------------
# 2. Cargar diagnósticos
# ------------------------------------------------------------

demand = pd.read_csv(
    demand_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

b2b = pd.read_csv(
    b2b_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

for data in [demand, b2b]:
    data["year"] = pd.to_numeric(data["year"], errors="coerce").astype("Int64")
    data["entidad_id"] = data["entidad_id"].astype(str).str.strip().str.zfill(2)
    data["municipio_id"] = data["municipio_id"].astype(str).str.strip().str.zfill(3)
    data["geo_key"] = data["entidad_id"] + data["municipio_id"]

print("\nDiagnóstico de demanda:")
print(f"Filas: {demand.shape[0]:,} | Columnas: {demand.shape[1]:,}")

print("\nDiagnóstico de oferta B2B:")
print(f"Filas: {b2b.shape[0]:,} | Columnas: {b2b.shape[1]:,}")


# ------------------------------------------------------------
# 3. Validar columnas mínimas
# ------------------------------------------------------------

required_demand_cols = [
    "year",
    "geo_key",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "industrial_demand_score",
    "industrial_demand_percentile",
    "industrial_demand_group",
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

required_b2b_cols = [
    "year",
    "geo_key",
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",
    "b2b_supply_score",
    "b2b_supply_percentile",
    "b2b_supply_group"
]

missing_demand_cols = [
    col for col in required_demand_cols
    if col not in demand.columns
]

missing_b2b_cols = [
    col for col in required_b2b_cols
    if col not in b2b.columns
]

if missing_demand_cols:
    raise ValueError(f"Faltan columnas en demanda: {missing_demand_cols}")

if missing_b2b_cols:
    raise ValueError(f"Faltan columnas en oferta B2B: {missing_b2b_cols}")


# ------------------------------------------------------------
# 4. Cruzar demanda + oferta B2B
# ------------------------------------------------------------

combined = demand[required_demand_cols].merge(
    b2b[required_b2b_cols],
    on=["year", "geo_key"],
    how="inner"
)

print("\nBase combinada:")
print(f"Filas: {combined.shape[0]:,}")
print(f"Columnas: {combined.shape[1]:,}")


# ------------------------------------------------------------
# 5. Crear perfiles cruzados
# ------------------------------------------------------------

def classify_demand_supply_profile(row):
    demand_p = row["industrial_demand_percentile"]
    b2b_p = row["b2b_supply_percentile"]

    if demand_p >= 0.75 and b2b_p < 0.50:
        return "High demand + low B2B supply"

    elif demand_p >= 0.75 and b2b_p >= 0.75:
        return "High demand + high B2B supply"

    elif demand_p < 0.50 and b2b_p >= 0.75:
        return "Low demand + high B2B supply"

    elif demand_p < 0.50 and b2b_p < 0.50:
        return "Low demand + low B2B supply"

    else:
        return "Middle cases"


combined["demand_supply_profile"] = combined.apply(
    classify_demand_supply_profile,
    axis=1
)

combined["demand_minus_b2b_supply_percentile"] = (
    combined["industrial_demand_percentile"]
    - combined["b2b_supply_percentile"]
)


# ------------------------------------------------------------
# 6. Crear tablas interesantes
# ------------------------------------------------------------

top_b2b_supply = (
    combined
    .sort_values(
        ["year", "b2b_supply_score"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

bottom_b2b_supply = (
    combined
    .sort_values(
        ["year", "b2b_supply_score"],
        ascending=[True, True]
    )
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

high_demand_low_b2b = (
    combined[
        (combined["industrial_demand_percentile"] >= 0.75) &
        (combined["b2b_supply_percentile"] < 0.50)
    ]
    .sort_values(
        [
            "year",
            "demand_minus_b2b_supply_percentile",
            "industrial_demand_percentile",
            "b2b_supply_percentile"
        ],
        ascending=[True, False, False, True]
    )
    .reset_index(drop=True)
)

high_demand_high_b2b = (
    combined[
        (combined["industrial_demand_percentile"] >= 0.75) &
        (combined["b2b_supply_percentile"] >= 0.75)
    ]
    .sort_values(
        [
            "year",
            "industrial_demand_percentile",
            "b2b_supply_percentile"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Tabla cruzada resumen
# ------------------------------------------------------------

cross_tab = (
    combined
    .groupby(["year", "demand_supply_profile"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_industrial_demand_score=("industrial_demand_score", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_b2b_supply_score=("b2b_supply_score", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_demand_minus_b2b_supply_percentile=("demand_minus_b2b_supply_percentile", "mean"),
        avg_manufacturing_establishments=("total_manufacturing_establishments", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_b2b_total_descriptive=("denue_b2b_support_establishments", "mean"),
        avg_logistics=("denue_logistics_storage_establishments", "mean"),
        avg_professional_technical=("denue_professional_technical_establishments", "mean"),
        avg_business_support=("denue_business_support_establishments", "mean"),
        avg_b2b_scian_classes=("denue_b2b_support_scian_classes", "mean")
    )
    .reset_index()
)

cross_tab["share_municipality_years"] = (
    cross_tab["n_municipality_years"] /
    cross_tab.groupby("year")["n_municipality_years"].transform("sum")
)

profile_order = {
    "High demand + low B2B supply": 1,
    "High demand + high B2B supply": 2,
    "Low demand + high B2B supply": 3,
    "Low demand + low B2B supply": 4,
    "Middle cases": 5
}

cross_tab["profile_order"] = (
    cross_tab["demand_supply_profile"]
    .map(profile_order)
)

cross_tab = (
    cross_tab
    .sort_values(["year", "profile_order"])
    .drop(columns=["profile_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Mostrar resultados
# ------------------------------------------------------------

print("\nTOP B2B SUPPLY")
print("=" * 100)
display(top_b2b_supply.head(20))

print("\nBOTTOM B2B SUPPLY")
print("=" * 100)
display(bottom_b2b_supply.head(20))

print("\nHIGH DEMAND + LOW B2B SUPPLY")
print("=" * 100)
display(high_demand_low_b2b.head(30))

print("\nHIGH DEMAND + HIGH B2B SUPPLY")
print("=" * 100)
display(high_demand_high_b2b.head(30))

print("\nCROSS TAB")
print("=" * 100)
display(cross_tab)

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas demanda:", len(demand))
print("Filas B2B:", len(b2b))
print("Filas combinadas:", len(combined))
print("Missing industrial_demand_percentile:", combined["industrial_demand_percentile"].isna().sum())
print("Missing b2b_supply_percentile:", combined["b2b_supply_percentile"].isna().sum())


Montando Google Drive en /content/drive...
Mounted at /content/drive
Google Drive montado correctamente en: /content/drive/MyDrive
Demand input:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/01_industrial_demand_score_diagnostic.csv

B2B input:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/scoring_diagnostics/05_b2b_supply_score_diagnostic.csv

Diagnóstico de demanda:
Filas: 4,530 | Columnas: 19

Diagnóstico de oferta B2B:
Filas: 4,530 | Columnas: 18

Base combinada:
Filas: 4,530
Columnas: 22

TOP B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile,demand_minus_b2b_supply_percentile
0,2018,14039,14,Jalisco,039,Guadalajara,0.995345,0.998664,Top industrial demand,8007.0,...,7008.0,817.0,3751.0,2440.0,110.0,0.999733,1.000000,Top B2B supply,High demand + high B2B supply,-0.001336
1,2018,02004,02,Baja California,004,Tijuana,0.996815,0.999109,Top industrial demand,3670.0,...,4275.0,782.0,1905.0,1588.0,105.0,0.998352,0.999555,Top B2B supply,High demand + high B2B supply,-0.000445
2,2018,21114,21,Puebla,114,Puebla,0.989087,0.993318,Top industrial demand,6849.0,...,5014.0,611.0,2303.0,2100.0,107.0,0.998330,0.999109,Top B2B supply,High demand + high B2B supply,-0.005791
3,2018,19039,19,Nuevo León,039,Monterrey,0.993964,0.996882,Top industrial demand,4729.0,...,4737.0,627.0,2543.0,1567.0,102.0,0.997973,0.998664,Top B2B supply,High demand + high B2B supply,-0.001782
4,2018,09015,09,Ciudad de México,015,Cuauhtémoc,0.976570,0.985746,Top industrial demand,3979.0,...,6122.0,459.0,4022.0,1641.0,99.0,0.997038,0.998218,Top B2B supply,High demand + high B2B supply,-0.012472
5,2018,11020,11,Guanajuato,020,León,0.992138,0.995546,Top industrial demand,10899.0,...,4220.0,539.0,1837.0,1844.0,98.0,0.996526,0.997773,Top B2B supply,High demand + high B2B supply,-0.002227
6,2018,22014,22,Querétaro,014,Querétaro,0.995122,0.998218,Top industrial demand,3363.0,...,3543.0,427.0,1809.0,1307.0,103.0,0.996080,0.997327,Top B2B supply,High demand + high B2B supply,0.000891
7,2018,24028,24,San Luis Potosí,028,San Luis Potosí,0.997127,0.999555,Top industrial demand,3824.0,...,3054.0,486.0,1448.0,1120.0,99.0,0.995278,0.996882,Top B2B supply,High demand + high B2B supply,0.002673
8,2018,09016,09,Ciudad de México,016,Miguel Hidalgo,0.972294,0.979955,Top industrial demand,1233.0,...,3945.0,353.0,2510.0,1082.0,104.0,0.995212,0.996437,Top B2B supply,High demand + high B2B supply,-0.016481
9,2018,31050,31,Yucatán,050,Mérida,0.976548,0.985301,Top industrial demand,3850.0,...,3121.0,412.0,1575.0,1134.0,100.0,0.995189,0.995991,Top B2B supply,High demand + high B2B supply,-0.010690



BOTTOM B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile,demand_minus_b2b_supply_percentile
0,2018,19016,19,Nuevo León,016,Doctor González,0.558018,0.559911,Medium industrial demand,8.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.537194
1,2018,17035,17,Morelos,035,Xoxocotla,0.560624,0.563920,Medium industrial demand,208.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.541203
2,2018,31016,31,Yucatán,016,Chacsinkín,0.576325,0.584855,Medium industrial demand,448.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.562138
3,2018,17034,17,Morelos,034,Coatetelco,0.475234,0.465479,Low industrial demand,74.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply,0.442762
4,2018,20497,20,Oaxaca,497,Santiago Yaitepec,0.569644,0.575501,Medium industrial demand,278.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.552784
5,2018,29003,29,Tlaxcala,003,Atlangatepec,0.491726,0.485078,Low industrial demand,7.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply,0.462361
6,2018,20328,20,Oaxaca,328,San Pedro Taviche,0.555223,0.556347,Medium industrial demand,296.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.533630
7,2018,20313,20,Oaxaca,313,San Pedro Jocotipac,0.520991,0.522049,Medium industrial demand,195.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.499332
8,2018,20533,20,Oaxaca,533,Santo Tomás Tamazulapan,0.425490,0.418263,Low industrial demand,15.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Low demand + low B2B supply,0.395546
9,2018,20564,20,Oaxaca,564,Yutanduchi de Guerrero,0.515212,0.513586,Medium industrial demand,264.0,...,0.0,0.0,0.0,0.0,0.0,0.098118,0.022717,Low B2B supply,Middle cases,0.490869



HIGH DEMAND + LOW B2B SUPPLY


...


HIGH DEMAND + HIGH B2B SUPPLY


,year,geo_key,entidad_id,entidad_name,municipio_id,municipio_name,industrial_demand_score,industrial_demand_percentile,industrial_demand_group,total_manufacturing_establishments,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,b2b_supply_score,b2b_supply_percentile,b2b_supply_group,demand_supply_profile,demand_minus_b2b_supply_percentile
0,2018,15106,15,México,106,Toluca,0.997305,1.000000,Top industrial demand,4351.0,...,2404.0,311.0,885.0,1208.0,92.0,0.990802,0.992428,Top B2B supply,High demand + high B2B supply,0.007572
1,2018,24028,24,San Luis Potosí,028,San Luis Potosí,0.997127,0.999555,Top industrial demand,3824.0,...,3054.0,486.0,1448.0,1120.0,99.0,0.995278,0.996882,Top B2B supply,High demand + high B2B supply,0.002673
2,2018,02004,02,Baja California,004,Tijuana,0.996815,0.999109,Top industrial demand,3670.0,...,4275.0,782.0,1905.0,1588.0,105.0,0.998352,0.999555,Top B2B supply,High demand + high B2B supply,-0.000445
3,2018,14039,14,Jalisco,039,Guadalajara,0.995345,0.998664,Top industrial demand,8007.0,...,7008.0,817.0,3751.0,2440.0,110.0,0.999733,1.000000,Top B2B supply,High demand + high B2B supply,-0.001336
4,2018,22014,22,Querétaro,014,Querétaro,0.995122,0.998218,Top industrial demand,3363.0,...,3543.0,427.0,1809.0,1307.0,103.0,0.996080,0.997327,Top B2B supply,High demand + high B2B supply,0.000891
5,2018,01001,01,Aguascalientes,001,Aguascalientes,0.995056,0.997773,Top industrial demand,3896.0,...,2938.0,352.0,1395.0,1191.0,99.0,0.994098,0.995546,Top B2B supply,High demand + high B2B supply,0.002227
6,2018,14120,14,Jalisco,120,Zapopan,0.994076,0.997327,Top industrial demand,4395.0,...,2728.0,334.0,1177.0,1217.0,101.0,0.993898,0.995100,Top B2B supply,High demand + high B2B supply,0.002227
7,2018,19039,19,Nuevo León,039,Monterrey,0.993964,0.996882,Top industrial demand,4729.0,...,4737.0,627.0,2543.0,1567.0,102.0,0.997973,0.998664,Top B2B supply,High demand + high B2B supply,-0.001782
8,2018,08037,08,Chihuahua,037,Juárez,0.993853,0.996437,Top industrial demand,2375.0,...,2135.0,553.0,799.0,783.0,93.0,0.992160,0.993764,Top B2B supply,High demand + high B2B supply,0.002673
9,2018,05030,05,Coahuila de Zaragoza,030,Saltillo,0.993831,0.995991,Top industrial demand,2654.0,...,1608.0,240.0,712.0,656.0,86.0,0.985089,0.987528,Top B2B supply,High demand + high B2B supply,0.008463



CROSS TAB


,year,demand_supply_profile,n_municipality_years,n_municipalities,avg_industrial_demand_score,avg_industrial_demand_percentile,avg_b2b_supply_score,avg_b2b_supply_percentile,avg_demand_minus_b2b_supply_percentile,avg_manufacturing_establishments,avg_manufacturing_employment,avg_b2b_total_descriptive,avg_logistics,avg_professional_technical,avg_business_support,avg_b2b_scian_classes,share_municipality_years
0,2018,High demand + low B2B supply,23,23,0.777090,0.808676,0.380611,0.393619,0.415058,394.086957,2040.826087,13.956522,0.695652,2.347826,10.913043,4.173913,0.010245
1,2018,High demand + high B2B supply,429,429,0.869226,0.894290,0.888273,0.895293,-0.001004,912.946387,13580.337995,506.440559,76.895105,222.174825,207.370629,41.550117,0.191091
2,2018,Low demand + high B2B supply,5,5,0.447517,0.437416,0.761236,0.775323,-0.337906,58.400000,138.200000,49.600000,12.400000,16.400000,20.800000,16.800000,0.002227
3,2018,Low demand + low B2B supply,944,944,0.242500,0.222661,0.249937,0.235095,-0.012434,33.556144,67.639831,4.417373,0.658898,0.679025,3.079449,2.398305,0.420490
4,2018,Middle cases,844,844,0.593686,0.602336,0.584633,0.597227,0.005109,173.268957,650.176540,30.889810,5.471564,8.849526,16.568720,10.002370,0.375947
5,2023,High demand + low B2B supply,30,30,0.771449,0.803807,0.381920,0.392560,0.411247,550.033333,2319.266667,11.833333,0.533333,3.233333,8.066667,4.533333,0.013129
6,2023,High demand + high B2B supply,436,436,0.864035,0.893709,0.889232,0.896240,-0.002531,978.694954,14813.130734,459.068807,81.133028,225.089450,152.846330,43.997706,0.190810
7,2023,Low demand + high B2B supply,6,6,0.407566,0.396426,0.763120,0.777899,-0.381473,56.833333,144.166667,45.500000,12.833333,15.333333,17.333333,18.333333,0.002626
8,2023,Low demand + low B2B supply,963,963,0.245205,0.222812,0.249678,0.233529,-0.010717,38.688474,77.427830,3.922118,0.595016,0.852544,2.474559,2.299065,0.421444
9,2023,Middle cases,850,850,0.593599,0.602685,0.586845,0.601067,0.001618,179.700000,688.152941,28.117647,5.029412,10.571765,12.516471,10.542353,0.371991



VALIDACIÓN RÁPIDA
Filas demanda: 4530
Filas B2B: 4530
Filas combinadas: 4530
Missing industrial_demand_percentile: 0
Missing b2b_supply_percentile: 0


In [ ]:
# ============================================================
# 07. DIAGNÓSTICO DEL SCALED B2B SUPPLY SCORE
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Construir y revisar en pantalla una medida de oferta B2B escalada.
#
# No guarda archivos.
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
#
# Lógica:
# b2b_scaled_capacity_proxy =
# 65.5 * denue_b2b_medium_establishments
# + 150 * denue_b2b_large_establishments
#
# scaled_b2b_supply_score =
# percentil anual de b2b_scaled_capacity_proxy
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

print("Input path:")
print(input_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
).astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Validar variables necesarias
# ------------------------------------------------------------

required_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias: {missing_cols}")

numeric_cols = [
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_numeric = df[numeric_cols].isna().sum()

print("\nMissing en variables B2B de tamaño:")
print(missing_numeric)

if missing_numeric.sum() > 0:
    raise ValueError(
        "Hay missing en variables B2B de tamaño. "
        "Revisa que estés usando la base complete-case correcta."
    )


# ------------------------------------------------------------
# 4. Construir proxy de capacidad B2B escalada
# ------------------------------------------------------------

medium_expected_workers = 65.5
large_expected_workers = 150

df["b2b_scaled_capacity_proxy"] = (
    medium_expected_workers * df["denue_b2b_medium_establishments"]
    + large_expected_workers * df["denue_b2b_large_establishments"]
)


# ------------------------------------------------------------
# 5. Normalizar por percentil anual
# ------------------------------------------------------------

df["scaled_b2b_supply_score"] = (
    df
    .groupby("year")["b2b_scaled_capacity_proxy"]
    .rank(pct=True, method="average")
)

df["scaled_b2b_supply_percentile"] = df["scaled_b2b_supply_score"]


# ------------------------------------------------------------
# 6. Crear grupo interpretable
# ------------------------------------------------------------

def classify_scaled_b2b_group(p):
    if pd.isna(p):
        return "Sin información"
    elif p >= 0.90:
        return "Top scaled B2B supply"
    elif p >= 0.75:
        return "High scaled B2B supply"
    elif p >= 0.50:
        return "Medium scaled B2B supply"
    else:
        return "Low scaled B2B supply"


df["scaled_b2b_supply_group"] = (
    df["scaled_b2b_supply_percentile"]
    .apply(classify_scaled_b2b_group)
)


# ------------------------------------------------------------
# 7. Crear variables auxiliares de atomización
# ------------------------------------------------------------

df["b2b_micro_small_establishments"] = (
    df["denue_b2b_micro_establishments"]
    + df["denue_b2b_small_establishments"]
)

df["b2b_micro_small_share"] = np.where(
    df["denue_b2b_support_establishments"] > 0,
    df["b2b_micro_small_establishments"] / df["denue_b2b_support_establishments"],
    np.nan
)

df["has_scaled_b2b_supply"] = (
    df["denue_b2b_medium_large_establishments"] > 0
).astype(int)

df["only_micro_small_b2b_supply"] = (
    (df["denue_b2b_support_establishments"] > 0) &
    (df["denue_b2b_medium_large_establishments"] == 0)
).astype(int)


# ------------------------------------------------------------
# 8. Tabla diagnóstica compacta
# ------------------------------------------------------------

scaled_diagnostic_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",
    "denue_b2b_support_establishments",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "b2b_micro_small_establishments",
    "b2b_micro_small_share",
    "has_scaled_b2b_supply",
    "only_micro_small_b2b_supply",
    "b2b_scaled_capacity_proxy",
    "scaled_b2b_supply_score",
    "scaled_b2b_supply_percentile",
    "scaled_b2b_supply_group"
]

scaled_diagnostic = df[scaled_diagnostic_cols].copy()


# ------------------------------------------------------------
# 9. Resumen por año y grupo
# ------------------------------------------------------------

scaled_summary = (
    scaled_diagnostic
    .groupby(["year", "scaled_b2b_supply_group"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_scaled_b2b_supply_score=("scaled_b2b_supply_score", "mean"),
        avg_scaled_capacity_proxy=("b2b_scaled_capacity_proxy", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium=("denue_b2b_medium_establishments", "mean"),
        avg_b2b_large=("denue_b2b_large_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_micro_small_share=("b2b_micro_small_share", "mean"),
        municipalities_with_scaled_b2b=("has_scaled_b2b_supply", "sum"),
        municipalities_only_micro_small=("only_micro_small_b2b_supply", "sum")
    )
    .reset_index()
)

scaled_summary["share_municipality_years"] = (
    scaled_summary["n_municipality_years"] /
    scaled_summary.groupby("year")["n_municipality_years"].transform("sum")
)

group_order = {
    "Top scaled B2B supply": 1,
    "High scaled B2B supply": 2,
    "Medium scaled B2B supply": 3,
    "Low scaled B2B supply": 4,
    "Sin información": 5
}

scaled_summary["group_order"] = (
    scaled_summary["scaled_b2b_supply_group"]
    .map(group_order)
)

scaled_summary = (
    scaled_summary
    .sort_values(["year", "group_order"])
    .drop(columns=["group_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Vistas interesantes
# ------------------------------------------------------------

top_scaled_b2b = (
    scaled_diagnostic
    .sort_values(
        ["year", "scaled_b2b_supply_score"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    .reset_index(drop=True)
)

bottom_scaled_b2b = (
    scaled_diagnostic
    .sort_values(
        ["year", "scaled_b2b_supply_score"],
        ascending=[True, True]
    )
    .groupby("year")
    .head(30)
    .reset_index(drop=True)
)

only_micro_small_cases = (
    scaled_diagnostic[
        scaled_diagnostic["only_micro_small_b2b_supply"] == 1
    ]
    .sort_values(
        ["year", "denue_b2b_support_establishments"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(30)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Mostrar resultados
# ------------------------------------------------------------

print("\nPARÁMETROS DE LA PROXY")
print("=" * 100)
print(f"Medium expected workers: {medium_expected_workers}")
print(f"Large expected workers: {large_expected_workers}")

print("\nRESUMEN DEL SCALED B2B SUPPLY SCORE")
print("=" * 100)
display(scaled_summary)

print("\nTOP SCALED B2B SUPPLY")
print("=" * 100)
display(top_scaled_b2b)

print("\nBOTTOM SCALED B2B SUPPLY")
print("=" * 100)
display(bottom_scaled_b2b)

print("\nCASOS CON B2B SOLO MICRO/SMALL")
print("=" * 100)
display(only_micro_small_cases)

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas diagnosticadas:", len(scaled_diagnostic))
print("Missing b2b_scaled_capacity_proxy:", scaled_diagnostic["b2b_scaled_capacity_proxy"].isna().sum())
print("Missing scaled_b2b_supply_score:", scaled_diagnostic["scaled_b2b_supply_score"].isna().sum())
print("Años:", sorted(scaled_diagnostic["year"].dropna().unique()))

Google Drive ya está disponible en: /content/drive/MyDrive
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Base cargada:
Filas: 4,530
Columnas: 54

Missing en variables B2B de tamaño:
denue_b2b_medium_establishments          0
denue_b2b_large_establishments           0
denue_b2b_medium_large_establishments    0
denue_b2b_support_establishments         0
denue_b2b_micro_establishments           0
denue_b2b_small_establishments           0
dtype: int64

PARÁMETROS DE LA PROXY
Medium expected workers: 65.5
Large expected workers: 150

RESUMEN DEL SCALED B2B SUPPLY SCORE


,year,scaled_b2b_supply_group,n_municipality_years,n_municipalities,avg_scaled_b2b_supply_score,avg_scaled_capacity_proxy,avg_b2b_total,avg_b2b_medium,avg_b2b_large,avg_b2b_medium_large,avg_micro_small_share,municipalities_with_scaled_b2b,municipalities_only_micro_small,share_municipality_years
0,2018,Top scaled B2B supply,225,225,0.950111,5511.453333,842.986667,35.706667,21.151111,56.857778,0.933456,225,0,0.100223
1,2018,High scaled B2B supply,247,247,0.844989,232.414980,101.089069,1.712551,0.801619,2.514170,0.953742,247,0,0.110022
2,2018,Medium scaled B2B supply,184,184,0.748998,65.500000,52.173913,1.000000,0.000000,1.000000,0.944743,184,0,0.081960
3,2018,Low scaled B2B supply,1589,1589,0.354120,0.000000,14.998112,0.000000,0.000000,0.000000,1.000000,0,1488,0.707795
4,2023,Top scaled B2B supply,229,229,0.950109,6280.600437,760.314410,39.585153,24.585153,64.170306,0.916977,229,0,0.100219
5,2023,High scaled B2B supply,286,286,0.837418,231.277972,88.643357,1.961538,0.685315,2.646853,0.938071,286,0,0.125164
6,2023,Medium scaled B2B supply,210,210,0.728884,65.500000,43.442857,1.000000,0.000000,1.000000,0.936142,210,0,0.091904
7,2023,Low scaled B2B supply,1560,1560,0.341575,0.000000,12.738462,0.000000,0.000000,0.000000,1.000000,0,1430,0.682713



TOP SCALED B2B SUPPLY


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_establishments,b2b_micro_small_share,has_scaled_b2b_supply,only_micro_small_b2b_supply,b2b_scaled_capacity_proxy,scaled_b2b_supply_score,scaled_b2b_supply_percentile,scaled_b2b_supply_group
0,2018,09,Ciudad de México,016,Miguel Hidalgo,09016,3945.0,1552.0,1544.0,539.0,310.0,849.0,3096.0,0.784791,1,0,81804.5,1.000000,1.000000,Top scaled B2B supply
1,2018,09,Ciudad de México,015,Cuauhtémoc,09015,6122.0,3722.0,1820.0,362.0,218.0,580.0,5542.0,0.905260,1,0,56411.0,0.999555,0.999555,Top scaled B2B supply
2,2018,19,Nuevo León,039,Monterrey,19039,4737.0,2999.0,1201.0,319.0,218.0,537.0,4200.0,0.886637,1,0,53594.5,0.999109,0.999109,Top scaled B2B supply
3,2018,09,Ciudad de México,014,Benito Juárez,09014,4296.0,2017.0,1697.0,404.0,178.0,582.0,3714.0,0.864525,1,0,53162.0,0.998664,0.998664,Top scaled B2B supply
4,2018,14,Jalisco,039,Guadalajara,14039,7008.0,4863.0,1690.0,294.0,161.0,455.0,6553.0,0.935074,1,0,43407.0,0.998218,0.998218,Top scaled B2B supply
5,2018,09,Ciudad de México,010,Álvaro Obregón,09010,1679.0,934.0,469.0,167.0,109.0,276.0,1403.0,0.835616,1,0,27288.5,0.997773,0.997773,Top scaled B2B supply
6,2018,22,Querétaro,014,Querétaro,22014,3543.0,2493.0,791.0,157.0,102.0,259.0,3284.0,0.926898,1,0,25583.5,0.997327,0.997327,Top scaled B2B supply
7,2018,11,Guanajuato,020,León,11020,4220.0,3307.0,681.0,132.0,100.0,232.0,3988.0,0.945024,1,0,23646.0,0.996882,0.996882,Top scaled B2B supply
8,2018,21,Puebla,114,Puebla,21114,5014.0,4017.0,748.0,163.0,86.0,249.0,4765.0,0.950339,1,0,23576.5,0.996437,0.996437,Top scaled B2B supply
9,2018,02,Baja California,004,Tijuana,02004,4275.0,3264.0,773.0,158.0,80.0,238.0,4037.0,0.944327,1,0,22349.0,0.995991,0.995991,Top scaled B2B supply



BOTTOM SCALED B2B SUPPLY


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_establishments,b2b_micro_small_share,has_scaled_b2b_supply,only_micro_small_b2b_supply,b2b_scaled_capacity_proxy,scaled_b2b_supply_score,scaled_b2b_supply_percentile,scaled_b2b_supply_group
0,2018,15,México,101,Tianguistenco,15101,167.0,159.0,8.0,0.0,0.0,0.0,167.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
1,2018,29,Tlaxcala,013,Huamantla,29013,174.0,163.0,11.0,0.0,0.0,0.0,174.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
2,2018,14,Jalisco,008,Arandas,14008,187.0,176.0,11.0,0.0,0.0,0.0,187.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
3,2018,11,Guanajuato,004,Apaseo el Alto,11004,111.0,106.0,5.0,0.0,0.0,0.0,111.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
4,2018,31,Yucatán,041,Kanasín,31041,57.0,49.0,8.0,0.0,0.0,0.0,57.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
5,2018,30,Veracruz de Ignacio de la Llave,123,Pánuco,30123,112.0,101.0,11.0,0.0,0.0,0.0,112.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
6,2018,14,Jalisco,083,Tala,14083,109.0,97.0,12.0,0.0,0.0,0.0,109.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
7,2018,31,Yucatán,038,Hunucmá,31038,41.0,40.0,1.0,0.0,0.0,0.0,41.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
8,2018,29,Tlaxcala,018,Contla de Juan Cuamatzi,29018,76.0,75.0,1.0,0.0,0.0,0.0,76.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
9,2018,22,Querétaro,007,Ezequiel Montes,22007,99.0,96.0,3.0,0.0,0.0,0.0,99.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply



CASOS CON B2B SOLO MICRO/SMALL


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,denue_b2b_support_establishments,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_establishments,b2b_micro_small_share,has_scaled_b2b_supply,only_micro_small_b2b_supply,b2b_scaled_capacity_proxy,scaled_b2b_supply_score,scaled_b2b_supply_percentile,scaled_b2b_supply_group
0,2018,20,Oaxaca,039,Heroica Ciudad de Huajuapan de León,20039,375.0,351.0,24.0,0.0,0.0,0.0,375.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
1,2018,15,México,120,Zumpango,15120,319.0,294.0,25.0,0.0,0.0,0.0,319.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
2,2018,09,Ciudad de México,009,Milpa Alta,09009,236.0,234.0,2.0,0.0,0.0,0.0,236.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
3,2018,14,Jalisco,008,Arandas,14008,187.0,176.0,11.0,0.0,0.0,0.0,187.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
4,2018,29,Tlaxcala,013,Huamantla,29013,174.0,163.0,11.0,0.0,0.0,0.0,174.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
5,2018,15,México,088,Tenancingo,15088,169.0,160.0,9.0,0.0,0.0,0.0,169.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
6,2018,15,México,101,Tianguistenco,15101,167.0,159.0,8.0,0.0,0.0,0.0,167.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
7,2018,14,Jalisco,015,Autlán de Navarro,14015,159.0,141.0,18.0,0.0,0.0,0.0,159.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
8,2018,30,Veracruz de Ignacio de la Llave,160,Álamo Temapache,30160,142.0,138.0,4.0,0.0,0.0,0.0,142.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply
9,2018,07,Chiapas,017,Cintalapa de Figueroa,07017,136.0,120.0,16.0,0.0,0.0,0.0,136.0,1.0,0,1,0.0,0.354120,0.354120,Low scaled B2B supply



VALIDACIÓN RÁPIDA
Filas diagnosticadas: 4530
Missing b2b_scaled_capacity_proxy: 0
Missing scaled_b2b_supply_score: 0
Años: [np.int64(2018), np.int64(2023)]


In [ ]:
# ============================================================
# 08. DIAGNÓSTICO DEL SCORE DEFINITIVO DE OPORTUNIDAD
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
#
# NO GUARDA ARCHIVOS.
#
# Objetivo:
# Construir en memoria:
# 1. industrial_demand_score
# 2. b2b_supply_score
# 3. scaled_b2b_supply_score
# 4. industrial_b2b_opportunity_score
# 5. opportunity_category
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

print("Input path:")
print(input_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir variables y pesos
# ------------------------------------------------------------

# Demanda industrial
demand_features = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

demand_weights = {
    "total_manufacturing_establishments": 0.25,
    "total_manufacturing_employment": 0.20,
    "total_manufacturing_value_added": 0.25,
    "total_manufacturing_income": 0.20,
    "total_manufacturing_investment": 0.10
}

# Oferta B2B general
# No usamos denue_b2b_support_establishments para evitar doble conteo.
b2b_supply_features = [
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes"
]

b2b_supply_weights = {
    "denue_logistics_storage_establishments": 0.35,
    "denue_professional_technical_establishments": 0.25,
    "denue_business_support_establishments": 0.20,
    "denue_b2b_support_scian_classes": 0.20
}

# Oferta B2B escalada
scaled_features = [
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments"
]

required_cols = (
    ["year", "entidad_id", "entidad_name", "municipio_id", "municipio_name", "geo_key"]
    + demand_features
    + b2b_supply_features
    + scaled_features
)

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias: {missing_cols}")

if not np.isclose(sum(demand_weights.values()), 1.0):
    raise ValueError("Los pesos de demanda no suman 1.")

if not np.isclose(sum(b2b_supply_weights.values()), 1.0):
    raise ValueError("Los pesos de oferta B2B no suman 1.")


# ------------------------------------------------------------
# 4. Convertir variables a numéricas y validar missing
# ------------------------------------------------------------

numeric_cols = list(set(demand_features + b2b_supply_features + scaled_features))

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_numeric = df[numeric_cols].isna().sum()

print("\nMissing en variables usadas:")
print(missing_numeric[missing_numeric > 0])

if missing_numeric.sum() > 0:
    raise ValueError(
        "Hay missing en variables necesarias. "
        "Revisa que estés usando 08_modeling_base_complete_cases.csv."
    )


# ------------------------------------------------------------
# 5. Industrial demand score
# ------------------------------------------------------------

for col in demand_features:
    percentile_col = f"{col}_demand_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["industrial_demand_score"] = 0.0

for col, weight in demand_weights.items():
    df["industrial_demand_score"] += (
        df[f"{col}_demand_percentile"] * weight
    )

df["industrial_demand_percentile"] = (
    df
    .groupby("year")["industrial_demand_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 6. B2B supply score general
# ------------------------------------------------------------

for col in b2b_supply_features:
    percentile_col = f"{col}_b2b_supply_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["b2b_supply_score"] = 0.0

for col, weight in b2b_supply_weights.items():
    df["b2b_supply_score"] += (
        df[f"{col}_b2b_supply_percentile"] * weight
    )

df["b2b_supply_percentile"] = (
    df
    .groupby("year")["b2b_supply_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 7. Scaled B2B supply score
# ------------------------------------------------------------

medium_expected_workers = 65.5
large_expected_workers = 150

df["b2b_scaled_capacity_proxy"] = (
    medium_expected_workers * df["denue_b2b_medium_establishments"]
    + large_expected_workers * df["denue_b2b_large_establishments"]
)

df["scaled_b2b_supply_score"] = (
    df
    .groupby("year")["b2b_scaled_capacity_proxy"]
    .rank(pct=True, method="average")
)

df["scaled_b2b_supply_percentile"] = df["scaled_b2b_supply_score"]

# Variables auxiliares de atomización
df["b2b_micro_small_establishments"] = (
    df["denue_b2b_micro_establishments"]
    + df["denue_b2b_small_establishments"]
)

df["b2b_micro_small_share"] = np.where(
    df["denue_b2b_support_establishments"] > 0,
    df["b2b_micro_small_establishments"] / df["denue_b2b_support_establishments"],
    np.nan
)

df["only_micro_small_b2b_supply"] = (
    (df["denue_b2b_support_establishments"] > 0) &
    (df["denue_b2b_medium_large_establishments"] == 0)
).astype(int)


# ------------------------------------------------------------
# 8. Score definitivo de oportunidad
# ------------------------------------------------------------

df["industrial_b2b_opportunity_score"] = (
    df["industrial_demand_percentile"]
    * (1 - df["scaled_b2b_supply_percentile"])
)

df["industrial_b2b_opportunity_percentile"] = (
    df
    .groupby("year")["industrial_b2b_opportunity_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 9. Categorías interpretativas
# ------------------------------------------------------------

def classify_opportunity(row):
    demand_p = row["industrial_demand_percentile"]
    scaled_p = row["scaled_b2b_supply_percentile"]

    if demand_p >= 0.90 and scaled_p < 0.50:
        return "Top priority opportunity"

    elif demand_p >= 0.75 and scaled_p < 0.50:
        return "Priority opportunity"

    elif demand_p >= 0.75 and scaled_p >= 0.75:
        return "Industrial ecosystem hub"

    elif demand_p >= 0.50 and scaled_p < 0.75:
        return "Developing industrial market"

    elif demand_p < 0.50 and scaled_p >= 0.75:
        return "Service hub, lower industrial demand"

    elif demand_p < 0.50 and scaled_p < 0.75:
        return "Lower initial priority"

    else:
        return "Intermediate / monitor"


df["industrial_b2b_opportunity_category"] = (
    df
    .apply(classify_opportunity, axis=1)
)


# ------------------------------------------------------------
# 10. Tablas de revisión
# ------------------------------------------------------------

review_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    # Scores principales
    "industrial_demand_score",
    "industrial_demand_percentile",
    "b2b_supply_score",
    "b2b_supply_percentile",
    "b2b_scaled_capacity_proxy",
    "scaled_b2b_supply_score",
    "scaled_b2b_supply_percentile",
    "industrial_b2b_opportunity_score",
    "industrial_b2b_opportunity_percentile",
    "industrial_b2b_opportunity_category",

    # Variables manufactureras
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",

    # Variables B2B
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "b2b_micro_small_share",
    "only_micro_small_b2b_supply"
]

score_diagnostic = df[review_cols].copy()


# Ranking principal de oportunidades
priority_categories = [
    "Top priority opportunity",
    "Priority opportunity",
    "Developing industrial market"
]

opportunity_ranking = (
    score_diagnostic[
        score_diagnostic["industrial_b2b_opportunity_category"]
        .isin(priority_categories)
    ]
    .sort_values(
        [
            "year",
            "industrial_b2b_opportunity_category",
            "industrial_b2b_opportunity_score"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

# Top opportunity por score, sin filtrar categoría
top_opportunity_score = (
    score_diagnostic
    .sort_values(
        ["year", "industrial_b2b_opportunity_score"],
        ascending=[True, False]
    )
    .groupby("year")
    .head(40)
    .reset_index(drop=True)
)

# Hubs consolidados
ecosystem_hubs = (
    score_diagnostic[
        score_diagnostic["industrial_b2b_opportunity_category"]
        == "Industrial ecosystem hub"
    ]
    .sort_values(
        ["year", "industrial_demand_percentile", "scaled_b2b_supply_percentile"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Resumen por categoría
# ------------------------------------------------------------

category_summary = (
    score_diagnostic
    .groupby(["year", "industrial_b2b_opportunity_category"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_municipalities=("geo_key", "nunique"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_scaled_b2b_supply_percentile=("scaled_b2b_supply_percentile", "mean"),
        avg_opportunity_score=("industrial_b2b_opportunity_score", "mean"),
        avg_manufacturing_establishments=("total_manufacturing_establishments", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_micro_small_share=("b2b_micro_small_share", "mean")
    )
    .reset_index()
)

category_summary["share_municipality_years"] = (
    category_summary["n_municipality_years"]
    / category_summary.groupby("year")["n_municipality_years"].transform("sum")
)

category_order = {
    "Top priority opportunity": 1,
    "Priority opportunity": 2,
    "Developing industrial market": 3,
    "Industrial ecosystem hub": 4,
    "Service hub, lower industrial demand": 5,
    "Lower initial priority": 6,
    "Intermediate / monitor": 7
}

category_summary["category_order"] = (
    category_summary["industrial_b2b_opportunity_category"]
    .map(category_order)
)

category_summary = (
    category_summary
    .sort_values(["year", "category_order"])
    .drop(columns=["category_order"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Mostrar resultados
# ------------------------------------------------------------

print("\nFÓRMULA DEL SCORE DEFINITIVO")
print("=" * 100)
print("industrial_b2b_opportunity_score = industrial_demand_percentile * (1 - scaled_b2b_supply_percentile)")
print(f"Medium expected workers used for scaled proxy: {medium_expected_workers}")
print(f"Large expected workers used for scaled proxy: {large_expected_workers}")

print("\nRESUMEN POR CATEGORÍA")
print("=" * 100)
display(category_summary)

print("\nRANKING PRINCIPAL DE OPORTUNIDADES")
print("=" * 100)
display(opportunity_ranking.head(50))

print("\nTOP POR SCORE DE OPORTUNIDAD, SIN FILTRAR CATEGORÍA")
print("=" * 100)
display(top_opportunity_score.head(50))

print("\nINDUSTRIAL ECOSYSTEM HUBS")
print("=" * 100)
display(ecosystem_hubs.head(40))

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas diagnosticadas:", len(score_diagnostic))
print("Missing industrial_b2b_opportunity_score:", score_diagnostic["industrial_b2b_opportunity_score"].isna().sum())
print("Missing category:", score_diagnostic["industrial_b2b_opportunity_category"].isna().sum())
print("Años:", sorted(score_diagnostic["year"].dropna().unique()))

Google Drive ya está disponible en: /content/drive/MyDrive
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Base cargada:
Filas: 4,530
Columnas: 54

Missing en variables usadas:
Series([], dtype: int64)

FÓRMULA DEL SCORE DEFINITIVO
industrial_b2b_opportunity_score = industrial_demand_percentile * (1 - scaled_b2b_supply_percentile)
Medium expected workers used for scaled proxy: 65.5
Large expected workers used for scaled proxy: 150

RESUMEN POR CATEGORÍA


,year,industrial_b2b_opportunity_category,n_municipality_years,n_municipalities,avg_industrial_demand_percentile,avg_b2b_supply_percentile,avg_scaled_b2b_supply_percentile,avg_opportunity_score,avg_manufacturing_establishments,avg_manufacturing_employment,avg_b2b_total,avg_b2b_medium_large,avg_micro_small_share,share_municipality_years
0,2018,Top priority opportunity,6,6,0.932146,0.815293,0.354120,0.602054,539.666667,5892.000000,128.666667,0.000000,1.000000,0.002673
1,2018,Priority opportunity,139,139,0.812261,0.659260,0.354120,0.524623,381.913669,1674.302158,52.316547,0.000000,1.000000,0.061915
2,2018,Developing industrial market,525,525,0.644991,0.582144,0.457164,0.341398,200.024762,711.274286,34.432381,0.260952,0.991627,0.233853
3,2018,Industrial ecosystem hub,355,355,0.904461,0.899497,0.911278,0.077799,1011.135211,16040.160563,584.887324,36.912676,0.948508,0.158129
4,2018,"Service hub, lower industrial demand",19,19,0.357942,0.574036,0.821240,0.062593,46.421053,108.473684,22.631579,1.789474,0.872954,0.008463
5,2018,Lower initial priority,1103,1103,0.248254,0.286549,0.370946,0.154818,36.453309,76.639166,6.628286,0.042611,0.994219,0.491314
6,2018,Intermediate / monitor,98,98,0.654836,0.760761,0.850816,0.097166,179.040816,651.489796,67.102041,2.816327,0.941790,0.043653
7,2023,Top priority opportunity,10,10,0.930547,0.678096,0.341575,0.612695,674.100000,7372.900000,74.700000,0.000000,1.000000,0.004376
8,2023,Priority opportunity,124,124,0.812677,0.653739,0.341575,0.535087,437.725806,1765.056452,42.911290,0.000000,1.000000,0.054267
9,2023,Developing industrial market,525,525,0.643333,0.583001,0.459612,0.340244,208.718095,756.152381,31.074286,0.304762,0.989438,0.229759



RANKING PRINCIPAL DE OPORTUNIDADES


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_demand_score,industrial_demand_percentile,b2b_supply_score,b2b_supply_percentile,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply
0,2018,12,Guerrero,034,Huitzuco de los Figueroa,12034,0.722361,0.749220,0.770880,0.784410,...,57.0,10.0,13.0,34.0,15.0,0.0,0.0,0.0,1.0,1
1,2018,21,Puebla,170,Tepeyahualco,21170,0.721381,0.748330,0.177049,0.159911,...,4.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,1.0,1
2,2018,13,Hidalgo,082,Zapotlán de Juárez,13082,0.721347,0.747884,0.557127,0.571492,...,29.0,2.0,4.0,23.0,6.0,0.0,0.0,0.0,1.0,1
3,2018,20,Oaxaca,546,Teotitlán del Valle,20546,0.721314,0.747439,0.310535,0.319154,...,8.0,0.0,2.0,6.0,3.0,0.0,0.0,0.0,1.0,1
4,2018,12,Guerrero,042,Mártir de Cuilapan,12042,0.720412,0.746548,0.541849,0.554120,...,16.0,4.0,2.0,10.0,6.0,0.0,0.0,0.0,1.0,1
5,2018,07,Chiapas,003,Acapetahua,07003,0.720390,0.746102,0.640991,0.656570,...,31.0,3.0,16.0,12.0,10.0,0.0,0.0,0.0,1.0,1
6,2018,20,Oaxaca,021,Cosolapa,20021,0.719588,0.745212,0.566169,0.583073,...,23.0,2.0,10.0,11.0,6.0,0.0,0.0,0.0,1.0,1
7,2018,14,Jalisco,123,Zapotlán del Rey,14123,0.719220,0.744766,0.620256,0.640535,...,21.0,10.0,5.0,6.0,7.0,0.0,0.0,0.0,1.0,1
8,2018,30,Veracruz de Ignacio de la Llave,047,Coscomatepec,30047,0.716292,0.742094,0.623497,0.644543,...,34.0,2.0,8.0,24.0,11.0,0.0,0.0,0.0,1.0,1
9,2018,21,Puebla,045,Chalchicomula de Sesma,21045,0.713408,0.740757,0.811370,0.821826,...,105.0,8.0,50.0,47.0,18.0,0.0,0.0,0.0,1.0,1



TOP POR SCORE DE OPORTUNIDAD, SIN FILTRAR CATEGORÍA


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_demand_score,industrial_demand_percentile,b2b_supply_score,b2b_supply_percentile,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply
0,2018,15,México,101,Tianguistenco,15101,0.953296,0.968374,0.806013,0.814699,...,167.0,5.0,57.0,105.0,21.0,0.0,0.0,0.0,1.0,1
1,2018,29,Tlaxcala,013,Huamantla,29013,0.938140,0.956793,0.879866,0.891314,...,174.0,17.0,52.0,105.0,29.0,0.0,0.0,0.0,1.0,1
2,2018,14,Jalisco,008,Arandas,14008,0.911759,0.936303,0.846247,0.854343,...,187.0,8.0,108.0,71.0,28.0,0.0,0.0,0.0,1.0,1
3,2018,11,Guanajuato,004,Apaseo el Alto,11004,0.893185,0.918486,0.853207,0.861024,...,111.0,16.0,42.0,53.0,24.0,0.0,0.0,0.0,1.0,1
4,2018,31,Yucatán,041,Kanasín,31041,0.880267,0.909131,0.751269,0.766592,...,57.0,12.0,5.0,40.0,17.0,0.0,0.0,0.0,1.0,1
5,2018,29,Tlaxcala,018,Contla de Juan Cuamatzi,29018,0.875657,0.903786,0.686448,0.703786,...,76.0,3.0,9.0,64.0,12.0,0.0,0.0,0.0,1.0,1
6,2018,13,Hidalgo,081,Zacualtipán de Ángeles,13081,0.869510,0.899332,0.752116,0.767929,...,71.0,5.0,21.0,45.0,15.0,0.0,0.0,0.0,1.0,1
7,2018,31,Yucatán,038,Hunucmá,31038,0.868976,0.897550,0.659878,0.678396,...,41.0,3.0,16.0,22.0,9.0,0.0,0.0,0.0,1.0,1
8,2018,14,Jalisco,083,Tala,14083,0.868029,0.896659,0.843630,0.852116,...,109.0,14.0,37.0,58.0,22.0,0.0,0.0,0.0,1.0,1
9,2018,04,Campeche,001,Calkiní,04001,0.864120,0.893987,0.787661,0.797773,...,73.0,10.0,14.0,49.0,17.0,0.0,0.0,0.0,1.0,1



INDUSTRIAL ECOSYSTEM HUBS


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_demand_score,industrial_demand_percentile,b2b_supply_score,b2b_supply_percentile,...,denue_b2b_support_establishments,denue_logistics_storage_establishments,denue_professional_technical_establishments,denue_business_support_establishments,denue_b2b_support_scian_classes,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply
0,2018,15,México,106,Toluca,15106,0.997305,1.000000,0.990802,0.992428,...,2404.0,311.0,885.0,1208.0,92.0,75.0,71.0,146.0,0.939268,0
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.997127,0.999555,0.995278,0.996882,...,3054.0,486.0,1448.0,1120.0,99.0,125.0,89.0,214.0,0.929928,0
2,2018,02,Baja California,004,Tijuana,02004,0.996815,0.999109,0.998352,0.999555,...,4275.0,782.0,1905.0,1588.0,105.0,158.0,80.0,238.0,0.944327,0
3,2018,14,Jalisco,039,Guadalajara,14039,0.995345,0.998664,0.999733,1.000000,...,7008.0,817.0,3751.0,2440.0,110.0,294.0,161.0,455.0,0.935074,0
4,2018,22,Querétaro,014,Querétaro,22014,0.995122,0.998218,0.996080,0.997327,...,3543.0,427.0,1809.0,1307.0,103.0,157.0,102.0,259.0,0.926898,0
5,2018,01,Aguascalientes,001,Aguascalientes,01001,0.995056,0.997773,0.994098,0.995546,...,2938.0,352.0,1395.0,1191.0,99.0,99.0,41.0,140.0,0.952349,0
6,2018,14,Jalisco,120,Zapopan,14120,0.994076,0.997327,0.993898,0.995100,...,2728.0,334.0,1177.0,1217.0,101.0,127.0,63.0,190.0,0.930352,0
7,2018,19,Nuevo León,039,Monterrey,19039,0.993964,0.996882,0.997973,0.998664,...,4737.0,627.0,2543.0,1567.0,102.0,319.0,218.0,537.0,0.886637,0
8,2018,08,Chihuahua,037,Juárez,08037,0.993853,0.996437,0.992160,0.993764,...,2135.0,553.0,799.0,783.0,93.0,130.0,67.0,197.0,0.907728,0
9,2018,05,Coahuila de Zaragoza,030,Saltillo,05030,0.993831,0.995991,0.985089,0.987528,...,1608.0,240.0,712.0,656.0,86.0,70.0,46.0,116.0,0.927861,0



VALIDACIÓN RÁPIDA
Filas diagnosticadas: 4530
Missing industrial_b2b_opportunity_score: 0
Missing category: 0
Años: [np.int64(2018), np.int64(2023)]


In [ ]:
# ============================================================
# RANKING LIMPIO DEL SCORE DEFINITIVO
# NO GUARDA ARCHIVOS
# Requiere que ya exista score_diagnostic en memoria
# ============================================================

ranking_cols = [
    "year",
    "entidad_name",
    "municipio_name",
    "geo_key",
    "industrial_b2b_opportunity_score",
    "industrial_b2b_opportunity_percentile",
    "industrial_b2b_opportunity_category",
    "industrial_demand_percentile",
    "scaled_b2b_supply_percentile",
    "b2b_supply_percentile",
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "denue_b2b_support_establishments",
    "denue_b2b_medium_large_establishments",
    "b2b_micro_small_share",
    "only_micro_small_b2b_supply"
]

ranking_cols = [col for col in ranking_cols if col in score_diagnostic.columns]

# Ranking puro por score, por año
opportunity_score_ranking = (
    score_diagnostic[ranking_cols]
    .sort_values(
        ["year", "industrial_b2b_opportunity_score"],
        ascending=[True, False]
    )
    .copy()
)

opportunity_score_ranking["rank_opportunity_score_within_year"] = (
    opportunity_score_ranking
    .groupby("year")
    .cumcount() + 1
)

# Mostrar top 50 por año
top_opportunity_score_ranking = (
    opportunity_score_ranking
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

print("RANKING PURO POR INDUSTRIAL_B2B_OPPORTUNITY_SCORE")
print("=" * 100)
display(top_opportunity_score_ranking)


# Ranking restringido a categorías interpretables de oportunidad
opportunity_categories = [
    "Top priority opportunity",
    "Priority opportunity",
    "Developing industrial market"
]

opportunity_only_ranking = (
    opportunity_score_ranking[
        opportunity_score_ranking["industrial_b2b_opportunity_category"]
        .isin(opportunity_categories)
    ]
    .copy()
)

opportunity_only_ranking["rank_opportunity_within_year"] = (
    opportunity_only_ranking
    .groupby("year")
    .cumcount() + 1
)

print("\nRANKING SOLO DE CATEGORÍAS DE OPORTUNIDAD")
print("=" * 100)
display(opportunity_only_ranking.groupby("year").head(50).reset_index(drop=True))


# Ranking por categoría, pero ordenado correctamente por score dentro de cada categoría
category_score_ranking = (
    opportunity_score_ranking
    .sort_values(
        [
            "year",
            "industrial_b2b_opportunity_category",
            "industrial_b2b_opportunity_score"
        ],
        ascending=[True, True, False]
    )
    .copy()
)

category_score_ranking["rank_within_category_year"] = (
    category_score_ranking
    .groupby(["year", "industrial_b2b_opportunity_category"])
    .cumcount() + 1
)

print("\nTOP 20 POR CATEGORÍA Y AÑO")
print("=" * 100)

for year in sorted(category_score_ranking["year"].dropna().unique()):
    print(f"\nAÑO {year}")
    for category in category_score_ranking["industrial_b2b_opportunity_category"].dropna().unique():
        temp = category_score_ranking[
            (category_score_ranking["year"] == year) &
            (category_score_ranking["industrial_b2b_opportunity_category"] == category)
        ].head(20)

        if len(temp) > 0:
            print(f"\n{category}")
            display(temp)

RANKING PURO POR INDUSTRIAL_B2B_OPPORTUNITY_SCORE


,year,entidad_name,municipio_name,geo_key,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_category,industrial_demand_percentile,scaled_b2b_supply_percentile,b2b_supply_percentile,total_manufacturing_establishments,total_manufacturing_employment,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,rank_opportunity_score_within_year
0,2018,México,Tianguistenco,15101,0.625453,1.000000,Top priority opportunity,0.968374,0.354120,0.814699,671.0,10225.0,167.0,0.0,1.0,1,1
1,2018,Tlaxcala,Huamantla,29013,0.617973,0.999555,Top priority opportunity,0.956793,0.354120,0.891314,743.0,7179.0,174.0,0.0,1.0,1,2
2,2018,Jalisco,Arandas,14008,0.604739,0.999109,Top priority opportunity,0.936303,0.354120,0.854343,458.0,6485.0,187.0,0.0,1.0,1,3
3,2018,Guanajuato,Apaseo el Alto,11004,0.593231,0.998664,Top priority opportunity,0.918486,0.354120,0.861024,404.0,3985.0,111.0,0.0,1.0,1,4
4,2018,Yucatán,Kanasín,31041,0.587190,0.998218,Top priority opportunity,0.909131,0.354120,0.766592,318.0,3541.0,57.0,0.0,1.0,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2023,Guanajuato,Yuriria,11046,0.554978,0.980306,Priority opportunity,0.842888,0.341575,0.850766,341.0,1268.0,111.0,0.0,1.0,1,46
96,2023,Veracruz de Ignacio de la Llave,Carlos A. Carrillo,30208,0.554402,0.979869,Priority opportunity,0.842013,0.341575,0.496718,116.0,2024.0,14.0,0.0,1.0,1,47
97,2023,Guerrero,Técpan de Galeana,12057,0.552097,0.979431,Priority opportunity,0.838512,0.341575,0.862582,514.0,1471.0,86.0,0.0,1.0,1,48
98,2023,Veracruz de Ignacio de la Llave,San Rafael,30211,0.551232,0.978993,Priority opportunity,0.837199,0.341575,0.747046,129.0,943.0,41.0,0.0,1.0,1,49



RANKING SOLO DE CATEGORÍAS DE OPORTUNIDAD


,year,entidad_name,municipio_name,geo_key,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_category,industrial_demand_percentile,scaled_b2b_supply_percentile,b2b_supply_percentile,total_manufacturing_establishments,total_manufacturing_employment,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,rank_opportunity_score_within_year,rank_opportunity_within_year
0,2018,México,Tianguistenco,15101,0.625453,1.000000,Top priority opportunity,0.968374,0.354120,0.814699,671.0,10225.0,167.0,0.0,1.0,1,1,1
1,2018,Tlaxcala,Huamantla,29013,0.617973,0.999555,Top priority opportunity,0.956793,0.354120,0.891314,743.0,7179.0,174.0,0.0,1.0,1,2,2
2,2018,Jalisco,Arandas,14008,0.604739,0.999109,Top priority opportunity,0.936303,0.354120,0.854343,458.0,6485.0,187.0,0.0,1.0,1,3,3
3,2018,Guanajuato,Apaseo el Alto,11004,0.593231,0.998664,Top priority opportunity,0.918486,0.354120,0.861024,404.0,3985.0,111.0,0.0,1.0,1,4,4
4,2018,Yucatán,Kanasín,31041,0.587190,0.998218,Top priority opportunity,0.909131,0.354120,0.766592,318.0,3541.0,57.0,0.0,1.0,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2023,Guanajuato,Yuriria,11046,0.554978,0.980306,Priority opportunity,0.842888,0.341575,0.850766,341.0,1268.0,111.0,0.0,1.0,1,46,46
96,2023,Veracruz de Ignacio de la Llave,Carlos A. Carrillo,30208,0.554402,0.979869,Priority opportunity,0.842013,0.341575,0.496718,116.0,2024.0,14.0,0.0,1.0,1,47,47
97,2023,Guerrero,Técpan de Galeana,12057,0.552097,0.979431,Priority opportunity,0.838512,0.341575,0.862582,514.0,1471.0,86.0,0.0,1.0,1,48,48
98,2023,Veracruz de Ignacio de la Llave,San Rafael,30211,0.551232,0.978993,Priority opportunity,0.837199,0.341575,0.747046,129.0,943.0,41.0,0.0,1.0,1,49,49



TOP 20 POR CATEGORÍA Y AÑO

AÑO 2018

Developing industrial market


...

,year,entidad_name,municipio_name,geo_key,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_category,industrial_demand_percentile,scaled_b2b_supply_percentile,b2b_supply_percentile,total_manufacturing_establishments,total_manufacturing_employment,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,rank_opportunity_score_within_year,rank_within_category_year
2515,2023,Hidalgo,Zacualtipán de Ángeles,13081,0.592150,0.995624,Priority opportunity,0.899344,0.341575,0.803939,363.0,3258.0,74.0,0.0,1.0,1,11,1
2494,2023,Guanajuato,San Felipe,11030,0.589268,0.995186,Priority opportunity,0.894967,0.341575,0.866958,347.0,4315.0,100.0,0.0,1.0,1,12,2
2500,2023,Tlaxcala,Tlaxco,29034,0.587251,0.994748,Priority opportunity,0.891904,0.341575,0.779869,343.0,3007.0,46.0,0.0,1.0,1,13,3
2497,2023,Aguascalientes,Rincón de Romos,01007,0.585522,0.994311,Priority opportunity,0.889278,0.341575,0.695405,226.0,2767.0,57.0,0.0,1.0,1,14,4
2524,2023,Guanajuato,Abasolo,11001,0.581488,0.993873,Priority opportunity,0.883151,0.341575,0.863020,286.0,2269.0,88.0,0.0,1.0,1,15,5
2635,2023,Yucatán,Tekit,31080,0.580336,0.993435,Priority opportunity,0.881400,0.341575,0.426477,1156.0,2782.0,7.0,0.0,1.0,1,16,6
2551,2023,Tlaxcala,Tepetitla de Lardizábal,29019,0.579471,0.992998,Priority opportunity,0.880088,0.341575,0.545295,295.0,1708.0,29.0,0.0,1.0,1,17,7
2670,2023,Yucatán,Tekax,31079,0.579183,0.992560,Priority opportunity,0.879650,0.341575,0.770241,1171.0,3166.0,55.0,0.0,1.0,1,18,8
2731,2023,Michoacán de Ocampo,Nahuatzen,16056,0.578030,0.992123,Priority opportunity,0.877899,0.341575,0.503282,1748.0,2873.0,39.0,0.0,1.0,1,19,9
2531,2023,Tlaxcala,La Magdalena Tlaltelulco,29048,0.577166,0.991685,Priority opportunity,0.876586,0.341575,0.590372,242.0,1990.0,25.0,0.0,1.0,1,20,10



Service hub, lower industrial demand


,year,entidad_name,municipio_name,geo_key,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_category,industrial_demand_percentile,scaled_b2b_supply_percentile,b2b_supply_percentile,total_manufacturing_establishments,total_manufacturing_employment,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,rank_opportunity_score_within_year,rank_within_category_year
3456,2023,Guerrero,Tecoanapa,12056,0.102138,0.308534,"Service hub, lower industrial demand",0.489278,0.791247,0.470897,90.0,190.0,13.0,2.0,0.846154,0,1581,1
3369,2023,Chiapas,Juárez,07048,0.099123,0.298031,"Service hub, lower industrial demand",0.474836,0.791247,0.641138,83.0,200.0,19.0,2.0,0.894737,0,1605,2
3400,2023,Campeche,Candelaria,04011,0.097113,0.291028,"Service hub, lower industrial demand",0.465208,0.791247,0.755361,73.0,191.0,35.0,2.0,0.942857,0,1621,3
3242,2023,Veracruz de Ignacio de la Llave,Angel R. Cabada,30015,0.089374,0.265208,"Service hub, lower industrial demand",0.496280,0.819912,0.736543,96.0,241.0,32.0,1.0,0.968750,0,1680,4
3568,2023,México,Villa de Allende,15111,0.088800,0.262582,"Service hub, lower industrial demand",0.425383,0.791247,0.587746,58.0,129.0,18.0,2.0,0.888889,0,1686,5
3530,2023,Chiapas,Socoltenango,07083,0.087338,0.258206,"Service hub, lower industrial demand",0.418381,0.791247,0.568490,62.0,140.0,14.0,2.0,0.857143,0,1696,6
3426,2023,Tamaulipas,Xicoténcatl,28043,0.084015,0.246389,"Service hub, lower industrial demand",0.466521,0.819912,0.709409,74.0,155.0,27.0,1.0,0.962963,0,1723,7
3674,2023,Guerrero,La Unión de Isidoro Montes de Oca,12068,0.083501,0.244201,"Service hub, lower industrial demand",0.400000,0.791247,0.503720,48.0,103.0,8.0,2.0,0.750000,0,1728,8
3627,2023,Nuevo León,General Bravo,19020,0.081857,0.238074,"Service hub, lower industrial demand",0.392123,0.791247,0.538293,24.0,80.0,12.0,2.0,0.833333,0,1742,9
3513,2023,Veracruz de Ignacio de la Llave,Atzacan,30022,0.080389,0.232385,"Service hub, lower industrial demand",0.446389,0.819912,0.522976,73.0,162.0,14.0,1.0,0.928571,0,1755,10



Top priority opportunity


,year,entidad_name,municipio_name,geo_key,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_category,industrial_demand_percentile,scaled_b2b_supply_percentile,b2b_supply_percentile,total_manufacturing_establishments,total_manufacturing_employment,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,rank_opportunity_score_within_year,rank_within_category_year
2356,2023,Tlaxcala,Huamantla,29013,0.631050,1.000000,Top priority opportunity,0.958425,0.341575,0.892779,856.0,8708.0,158.0,0.0,1.0,1,1,1
2326,2023,San Luis Potosí,Villa de Pozos,24059,0.630762,0.999562,Top priority opportunity,0.957987,0.341575,0.028665,460.0,15911.0,0.0,0.0,NaN,0,2,2
2376,2023,Yucatán,Kanasín,31041,0.622694,0.999125,Top priority opportunity,0.945733,0.341575,0.844201,563.0,8973.0,68.0,0.0,1.0,1,3,3
2403,2023,Jalisco,Arandas,14008,0.622406,0.998687,Top priority opportunity,0.945295,0.341575,0.884902,502.0,7271.0,219.0,0.0,1.0,1,4,4
2393,2023,Tlaxcala,Papalotla de Xicohténcatl,29041,0.619812,0.998249,Top priority opportunity,0.941357,0.341575,0.720788,461.0,9879.0,36.0,0.0,1.0,1,5,5
2434,2023,Tlaxcala,Teolocholco,29028,0.603100,0.997812,Top priority opportunity,0.915974,0.341575,0.637199,252.0,6529.0,43.0,0.0,1.0,1,6,6
2523,2023,Puebla,Ajalpan,21010,0.602235,0.997374,Top priority opportunity,0.914661,0.341575,0.751422,2288.0,7759.0,63.0,0.0,1.0,1,7,7
2508,2023,Tlaxcala,Contla de Juan Cuamatzi,29018,0.599065,0.996937,Top priority opportunity,0.909847,0.341575,0.683589,713.0,3501.0,63.0,0.0,1.0,1,8,8
2469,2023,Tlaxcala,Ixtacuixtla de Mariano Matamoros,29015,0.598489,0.996499,Top priority opportunity,0.908972,0.341575,0.546171,371.0,3145.0,44.0,0.0,1.0,1,9,9
2505,2023,Veracruz de Ignacio de la Llave,Tres Valles,30207,0.597337,0.996061,Top priority opportunity,0.907221,0.341575,0.791247,275.0,2053.0,53.0,0.0,1.0,1,10,10


In [ ]:
# ============================================================
# 09. SCORE DEFINITIVO V2: AJUSTE DE CEROS EN OFERTA B2B ESCALADA
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
# NO GUARDA ARCHIVOS.
#
# Objetivo:
# Comparar:
# - V1: percentil tradicional con rank(pct=True, method="average")
# - V2: ceros explícitos en scaled B2B supply:
#       si b2b_scaled_capacity_proxy == 0 -> scaled_b2b_supply_percentile_v2 = 0
#       si b2b_scaled_capacity_proxy > 0  -> percentil anual entre positivos
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

print("Input path:")
print(input_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir variables y pesos
# ------------------------------------------------------------

demand_features = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

demand_weights = {
    "total_manufacturing_establishments": 0.25,
    "total_manufacturing_employment": 0.20,
    "total_manufacturing_value_added": 0.25,
    "total_manufacturing_income": 0.20,
    "total_manufacturing_investment": 0.10
}

b2b_supply_features = [
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes"
]

b2b_supply_weights = {
    "denue_logistics_storage_establishments": 0.35,
    "denue_professional_technical_establishments": 0.25,
    "denue_business_support_establishments": 0.20,
    "denue_b2b_support_scian_classes": 0.20
}

scaled_features = [
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments"
]

required_cols = (
    ["year", "entidad_id", "entidad_name", "municipio_id", "municipio_name", "geo_key"]
    + demand_features
    + b2b_supply_features
    + scaled_features
)

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias: {missing_cols}")

if not np.isclose(sum(demand_weights.values()), 1.0):
    raise ValueError("Los pesos de demanda no suman 1.")

if not np.isclose(sum(b2b_supply_weights.values()), 1.0):
    raise ValueError("Los pesos de oferta B2B no suman 1.")


# ------------------------------------------------------------
# 4. Convertir variables a numéricas
# ------------------------------------------------------------

numeric_cols = list(set(demand_features + b2b_supply_features + scaled_features))

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_numeric = df[numeric_cols].isna().sum()

if missing_numeric.sum() > 0:
    print("Missing detectados:")
    print(missing_numeric[missing_numeric > 0])
    raise ValueError(
        "Hay missing en variables necesarias. "
        "Revisa que estés usando 08_modeling_base_complete_cases.csv."
    )


# ------------------------------------------------------------
# 5. Industrial demand score
# ------------------------------------------------------------

for col in demand_features:
    percentile_col = f"{col}_demand_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["industrial_demand_score"] = 0.0

for col, weight in demand_weights.items():
    df["industrial_demand_score"] += (
        df[f"{col}_demand_percentile"] * weight
    )

df["industrial_demand_percentile"] = (
    df
    .groupby("year")["industrial_demand_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 6. B2B supply score general
# ------------------------------------------------------------

for col in b2b_supply_features:
    percentile_col = f"{col}_b2b_supply_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["b2b_supply_score"] = 0.0

for col, weight in b2b_supply_weights.items():
    df["b2b_supply_score"] += (
        df[f"{col}_b2b_supply_percentile"] * weight
    )

df["b2b_supply_percentile"] = (
    df
    .groupby("year")["b2b_supply_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 7. Oferta B2B escalada: proxy de capacidad
# ------------------------------------------------------------

medium_expected_workers = 65.5
large_expected_workers = 150

df["b2b_scaled_capacity_proxy"] = (
    medium_expected_workers * df["denue_b2b_medium_establishments"]
    + large_expected_workers * df["denue_b2b_large_establishments"]
)

df["has_scaled_b2b_supply"] = (
    df["b2b_scaled_capacity_proxy"] > 0
).astype(int)


# ------------------------------------------------------------
# 8. Scaled B2B supply V1: método anterior
# ------------------------------------------------------------

df["scaled_b2b_supply_percentile_v1"] = (
    df
    .groupby("year")["b2b_scaled_capacity_proxy"]
    .rank(pct=True, method="average")
)

df["industrial_b2b_opportunity_score_v1"] = (
    df["industrial_demand_percentile"]
    * (1 - df["scaled_b2b_supply_percentile_v1"])
)

df["industrial_b2b_opportunity_percentile_v1"] = (
    df
    .groupby("year")["industrial_b2b_opportunity_score_v1"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 9. Scaled B2B supply V2: ceros explícitos
# ------------------------------------------------------------

df["scaled_b2b_supply_percentile_v2"] = np.nan

for year_value, idx in df.groupby("year").groups.items():
    year_mask = df.index.isin(idx)
    positive_mask = year_mask & (df["b2b_scaled_capacity_proxy"] > 0)
    zero_mask = year_mask & (df["b2b_scaled_capacity_proxy"] == 0)

    # Ceros explícitos: ausencia de oferta escalada
    df.loc[zero_mask, "scaled_b2b_supply_percentile_v2"] = 0.0

    # Positivos: percentil anual solo entre municipios con oferta escalada positiva
    df.loc[positive_mask, "scaled_b2b_supply_percentile_v2"] = (
        df.loc[positive_mask, "b2b_scaled_capacity_proxy"]
        .rank(pct=True, method="average")
    )

df["industrial_b2b_opportunity_score_v2"] = (
    df["industrial_demand_percentile"]
    * (1 - df["scaled_b2b_supply_percentile_v2"])
)

df["industrial_b2b_opportunity_percentile_v2"] = (
    df
    .groupby("year")["industrial_b2b_opportunity_score_v2"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 10. Variables auxiliares de atomización
# ------------------------------------------------------------

df["b2b_micro_small_establishments"] = (
    df["denue_b2b_micro_establishments"]
    + df["denue_b2b_small_establishments"]
)

df["b2b_micro_small_share"] = np.where(
    df["denue_b2b_support_establishments"] > 0,
    df["b2b_micro_small_establishments"] / df["denue_b2b_support_establishments"],
    np.nan
)

df["only_micro_small_b2b_supply"] = (
    (df["denue_b2b_support_establishments"] > 0) &
    (df["denue_b2b_medium_large_establishments"] == 0)
).astype(int)


# ------------------------------------------------------------
# 11. Categorías con V2
# ------------------------------------------------------------

def classify_opportunity_v2(row):
    demand_p = row["industrial_demand_percentile"]
    scaled_p = row["scaled_b2b_supply_percentile_v2"]

    if demand_p >= 0.90 and scaled_p < 0.50:
        return "Top priority opportunity"

    elif demand_p >= 0.75 and scaled_p < 0.50:
        return "Priority opportunity"

    elif demand_p >= 0.75 and scaled_p >= 0.75:
        return "Industrial ecosystem hub"

    elif demand_p >= 0.50 and scaled_p < 0.75:
        return "Developing industrial market"

    elif demand_p < 0.50 and scaled_p >= 0.75:
        return "Service hub, lower industrial demand"

    elif demand_p < 0.50 and scaled_p < 0.75:
        return "Lower initial priority"

    else:
        return "Intermediate / monitor"


df["industrial_b2b_opportunity_category_v2"] = (
    df.apply(classify_opportunity_v2, axis=1)
)


# ------------------------------------------------------------
# 12. Subtipos de oportunidad con V2
# ------------------------------------------------------------

def classify_opportunity_subtype_v2(row):
    category = row["industrial_b2b_opportunity_category_v2"]
    b2b_general_p = row["b2b_supply_percentile"]
    b2b_total = row["denue_b2b_support_establishments"]
    medium_large = row["denue_b2b_medium_large_establishments"]

    opportunity_categories = [
        "Top priority opportunity",
        "Priority opportunity",
        "Developing industrial market"
    ]

    if category not in opportunity_categories:
        return "Not opportunity subtype"

    if medium_large == 0 and b2b_total > 0 and b2b_general_p >= 0.75:
        return "Atomized B2B opportunity"

    elif medium_large == 0 and b2b_total > 0 and b2b_general_p < 0.75:
        return "Limited scaled B2B opportunity"

    elif medium_large == 0 and b2b_total == 0:
        return "Broad B2B gap"

    elif medium_large > 0 and row["scaled_b2b_supply_percentile_v2"] < 0.50:
        return "Low scaled B2B opportunity"

    else:
        return "Mixed opportunity case"


df["industrial_b2b_opportunity_subtype_v2"] = (
    df.apply(classify_opportunity_subtype_v2, axis=1)
)


# ------------------------------------------------------------
# 13. Tabla diagnóstica compacta
# ------------------------------------------------------------

review_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    "industrial_demand_percentile",
    "b2b_supply_percentile",

    "b2b_scaled_capacity_proxy",
    "scaled_b2b_supply_percentile_v1",
    "scaled_b2b_supply_percentile_v2",

    "industrial_b2b_opportunity_score_v1",
    "industrial_b2b_opportunity_percentile_v1",
    "industrial_b2b_opportunity_score_v2",
    "industrial_b2b_opportunity_percentile_v2",

    "industrial_b2b_opportunity_category_v2",
    "industrial_b2b_opportunity_subtype_v2",

    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",

    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "b2b_micro_small_share",
    "only_micro_small_b2b_supply"
]

score_diagnostic_v2 = df[review_cols].copy()


# ------------------------------------------------------------
# 14. Rankings comparativos
# ------------------------------------------------------------

# Ranking V1
ranking_v1 = (
    score_diagnostic_v2
    .sort_values(
        ["year", "industrial_b2b_opportunity_score_v1"],
        ascending=[True, False]
    )
    .copy()
)

ranking_v1["rank_v1"] = (
    ranking_v1
    .groupby("year")
    .cumcount() + 1
)

# Ranking V2
ranking_v2 = (
    score_diagnostic_v2
    .sort_values(
        ["year", "industrial_b2b_opportunity_score_v2"],
        ascending=[True, False]
    )
    .copy()
)

ranking_v2["rank_v2"] = (
    ranking_v2
    .groupby("year")
    .cumcount() + 1
)

# Comparación de rankings
rank_compare = (
    ranking_v2[
        [
            "year",
            "geo_key",
            "entidad_name",
            "municipio_name",
            "rank_v2",
            "industrial_b2b_opportunity_score_v2",
            "industrial_b2b_opportunity_category_v2",
            "industrial_b2b_opportunity_subtype_v2",
            "industrial_demand_percentile",
            "scaled_b2b_supply_percentile_v2",
            "b2b_supply_percentile",
            "denue_b2b_support_establishments",
            "denue_b2b_medium_large_establishments",
            "b2b_micro_small_share"
        ]
    ]
    .merge(
        ranking_v1[["year", "geo_key", "rank_v1", "industrial_b2b_opportunity_score_v1"]],
        on=["year", "geo_key"],
        how="left"
    )
)

rank_compare["rank_change_v2_minus_v1"] = (
    rank_compare["rank_v2"] - rank_compare["rank_v1"]
)


# ------------------------------------------------------------
# 15. Resúmenes
# ------------------------------------------------------------

zero_summary = (
    df
    .groupby("year")
    .agg(
        n_municipality_years=("geo_key", "size"),
        n_zero_scaled_capacity=("has_scaled_b2b_supply", lambda x: (x == 0).sum()),
        n_positive_scaled_capacity=("has_scaled_b2b_supply", "sum"),
        avg_scaled_percentile_v1_for_zeros=(
            "scaled_b2b_supply_percentile_v1",
            lambda x: x[df.loc[x.index, "has_scaled_b2b_supply"] == 0].mean()
        ),
        avg_scaled_percentile_v2_for_zeros=(
            "scaled_b2b_supply_percentile_v2",
            lambda x: x[df.loc[x.index, "has_scaled_b2b_supply"] == 0].mean()
        )
    )
    .reset_index()
)

zero_summary["share_zero_scaled_capacity"] = (
    zero_summary["n_zero_scaled_capacity"] /
    zero_summary["n_municipality_years"]
)

category_summary_v2 = (
    score_diagnostic_v2
    .groupby(["year", "industrial_b2b_opportunity_category_v2"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        avg_opportunity_score_v2=("industrial_b2b_opportunity_score_v2", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_scaled_b2b_supply_percentile_v2=("scaled_b2b_supply_percentile_v2", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_manufacturing_establishments=("total_manufacturing_establishments", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_micro_small_share=("b2b_micro_small_share", "mean")
    )
    .reset_index()
)

subtype_summary_v2 = (
    score_diagnostic_v2
    .groupby(["year", "industrial_b2b_opportunity_subtype_v2"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        avg_opportunity_score_v2=("industrial_b2b_opportunity_score_v2", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_scaled_b2b_supply_percentile_v2=("scaled_b2b_supply_percentile_v2", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 16. Tablas de revisión
# ------------------------------------------------------------

top_v2 = (
    rank_compare
    .sort_values(["year", "rank_v2"])
    .groupby("year")
    .head(40)
    .reset_index(drop=True)
)

top_opportunity_categories_v2 = (
    rank_compare[
        rank_compare["industrial_b2b_opportunity_category_v2"].isin(
            [
                "Top priority opportunity",
                "Priority opportunity",
                "Developing industrial market"
            ]
        )
    ]
    .sort_values(["year", "rank_v2"])
    .groupby("year")
    .head(50)
    .reset_index(drop=True)
)

atomized_opportunities_v2 = (
    rank_compare[
        rank_compare["industrial_b2b_opportunity_subtype_v2"]
        == "Atomized B2B opportunity"
    ]
    .sort_values(["year", "rank_v2"])
    .groupby("year")
    .head(30)
    .reset_index(drop=True)
)

broad_gap_opportunities_v2 = (
    rank_compare[
        rank_compare["industrial_b2b_opportunity_subtype_v2"]
        == "Broad B2B gap"
    ]
    .sort_values(["year", "rank_v2"])
    .groupby("year")
    .head(30)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 17. Mostrar resultados
# ------------------------------------------------------------

print("\nAJUSTE METODOLÓGICO V2")
print("=" * 100)
print("V1: percentil tradicional con empates promedio.")
print("V2: b2b_scaled_capacity_proxy == 0 recibe scaled_b2b_supply_percentile_v2 = 0.")
print("Para valores positivos, el percentil se calcula dentro de los municipios con proxy positiva.")
print(f"Medium expected workers: {medium_expected_workers}")
print(f"Large expected workers: {large_expected_workers}")

print("\nRESUMEN DE CEROS EN OFERTA B2B ESCALADA")
print("=" * 100)
display(zero_summary)

print("\nRESUMEN POR CATEGORÍA V2")
print("=" * 100)
display(category_summary_v2)

print("\nRESUMEN POR SUBTIPO V2")
print("=" * 100)
display(subtype_summary_v2)

print("\nTOP RANKING V2 POR INDUSTRIAL_B2B_OPPORTUNITY_SCORE")
print("=" * 100)
display(top_v2)

print("\nTOP SOLO CATEGORÍAS DE OPORTUNIDAD V2")
print("=" * 100)
display(top_opportunity_categories_v2)

print("\nATOMIZED B2B OPPORTUNITIES V2")
print("=" * 100)
display(atomized_opportunities_v2)

print("\nBROAD B2B GAP OPPORTUNITIES V2")
print("=" * 100)
display(broad_gap_opportunities_v2)

print("\nVALIDACIÓN RÁPIDA")
print("=" * 100)
print("Filas diagnosticadas:", len(score_diagnostic_v2))
print("Missing scaled_b2b_supply_percentile_v2:", score_diagnostic_v2["scaled_b2b_supply_percentile_v2"].isna().sum())
print("Missing industrial_b2b_opportunity_score_v2:", score_diagnostic_v2["industrial_b2b_opportunity_score_v2"].isna().sum())
print("Años:", sorted(score_diagnostic_v2["year"].dropna().unique()))

Google Drive ya está disponible en: /content/drive/MyDrive
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Base cargada:
Filas: 4,530
Columnas: 54

AJUSTE METODOLÓGICO V2
V1: percentil tradicional con empates promedio.
V2: b2b_scaled_capacity_proxy == 0 recibe scaled_b2b_supply_percentile_v2 = 0.
Para valores positivos, el percentil se calcula dentro de los municipios con proxy positiva.
Medium expected workers: 65.5
Large expected workers: 150

RESUMEN DE CEROS EN OFERTA B2B ESCALADA


,year,n_municipality_years,n_zero_scaled_capacity,n_positive_scaled_capacity,avg_scaled_percentile_v1_for_zeros,avg_scaled_percentile_v2_for_zeros,share_zero_scaled_capacity
0,2018,2245,1589,656,0.354120,0.0,0.707795
1,2023,2285,1560,725,0.341575,0.0,0.682713



RESUMEN POR CATEGORÍA V2


,year,industrial_b2b_opportunity_category_v2,n_municipality_years,avg_opportunity_score_v2,avg_industrial_demand_percentile,avg_scaled_b2b_supply_percentile_v2,avg_b2b_supply_percentile,avg_manufacturing_establishments,avg_manufacturing_employment,avg_b2b_total,avg_b2b_medium_large,avg_micro_small_share
0,2018,Developing industrial market,672,0.516700,0.668647,0.193617,0.637374,243.049107,1363.879464,59.858631,1.337798,0.979746
1,2018,Industrial ecosystem hub,162,0.114277,0.944598,0.877113,0.953878,1565.012346,30009.592593,1086.413580,76.469136,0.927062
2,2018,Intermediate / monitor,2,0.165936,0.710579,0.766387,0.910913,295.500000,1458.000000,177.500000,11.000000,0.938024
3,2018,Lower initial priority,1122,0.245738,0.250111,0.012481,0.291417,36.622103,77.178253,6.899287,0.072193,0.991980
4,2018,Priority opportunity,258,0.710744,0.821189,0.132530,0.723372,377.321705,1761.395349,72.275194,0.658915,0.989785
5,2018,Top priority opportunity,29,0.713418,0.921780,0.226477,0.852915,794.310345,5660.862069,176.068966,1.310345,0.990035
6,2023,Developing industrial market,701,0.508602,0.670168,0.208940,0.645911,270.223966,1298.005706,55.748930,1.427960,0.975845
7,2023,Industrial ecosystem hub,184,0.117460,0.942461,0.873508,0.945881,1534.614130,30323.554348,897.070652,78.103261,0.910977
8,2023,Intermediate / monitor,1,0.149911,0.744420,0.798621,0.920350,216.000000,710.000000,175.000000,14.000000,0.920000
9,2023,Lower initial priority,1142,0.243359,0.250109,0.018300,0.289694,42.126970,88.134851,6.204028,0.091068,0.986549



RESUMEN POR SUBTIPO V2


...


ATOMIZED B2B OPPORTUNITIES V2


,year,geo_key,entidad_name,municipio_name,rank_v2,industrial_b2b_opportunity_score_v2,industrial_b2b_opportunity_category_v2,industrial_b2b_opportunity_subtype_v2,industrial_demand_percentile,scaled_b2b_supply_percentile_v2,b2b_supply_percentile,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,rank_v1,industrial_b2b_opportunity_score_v1,rank_change_v2_minus_v1
0,2018,15101,México,Tianguistenco,1,0.968374,Top priority opportunity,Atomized B2B opportunity,0.968374,0.0,0.814699,167.0,0.0,1.0,1,0.625453,0
1,2018,29013,Tlaxcala,Huamantla,2,0.956793,Top priority opportunity,Atomized B2B opportunity,0.956793,0.0,0.891314,174.0,0.0,1.0,2,0.617973,0
2,2018,14008,Jalisco,Arandas,3,0.936303,Top priority opportunity,Atomized B2B opportunity,0.936303,0.0,0.854343,187.0,0.0,1.0,3,0.604739,0
3,2018,11004,Guanajuato,Apaseo el Alto,4,0.918486,Top priority opportunity,Atomized B2B opportunity,0.918486,0.0,0.861024,111.0,0.0,1.0,4,0.593231,0
4,2018,31041,Yucatán,Kanasín,5,0.909131,Top priority opportunity,Atomized B2B opportunity,0.909131,0.0,0.766592,57.0,0.0,1.0,5,0.587190,0
5,2018,13081,Hidalgo,Zacualtipán de Ángeles,7,0.899332,Priority opportunity,Atomized B2B opportunity,0.899332,0.0,0.767929,71.0,0.0,1.0,7,0.580860,0
6,2018,14083,Jalisco,Tala,9,0.896659,Priority opportunity,Atomized B2B opportunity,0.896659,0.0,0.852116,109.0,0.0,1.0,9,0.579134,0
7,2018,04001,Campeche,Calkiní,10,0.893987,Priority opportunity,Atomized B2B opportunity,0.893987,0.0,0.797773,73.0,0.0,1.0,10,0.577408,0
8,2018,15120,México,Zumpango,11,0.891314,Priority opportunity,Atomized B2B opportunity,0.891314,0.0,0.901559,319.0,0.0,1.0,11,0.575682,0
9,2018,22007,Querétaro,Ezequiel Montes,12,0.889532,Priority opportunity,Atomized B2B opportunity,0.889532,0.0,0.838307,99.0,0.0,1.0,12,0.574531,0



BROAD B2B GAP OPPORTUNITIES V2


,year,geo_key,entidad_name,municipio_name,rank_v2,industrial_b2b_opportunity_score_v2,industrial_b2b_opportunity_category_v2,industrial_b2b_opportunity_subtype_v2,industrial_demand_percentile,scaled_b2b_supply_percentile_v2,b2b_supply_percentile,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,rank_v1,industrial_b2b_opportunity_score_v1,rank_change_v2_minus_v1
0,2018,31016,Yucatán,Chacsinkín,481,0.584855,Developing industrial market,Broad B2B gap,0.584855,0.0,0.022717,0.0,0.0,NaN,380,0.377746,101
1,2018,20497,Oaxaca,Santiago Yaitepec,499,0.575501,Developing industrial market,Broad B2B gap,0.575501,0.0,0.022717,0.0,0.0,NaN,395,0.371705,104
2,2018,17035,Morelos,Xoxocotla,530,0.563920,Developing industrial market,Broad B2B gap,0.563920,0.0,0.022717,0.0,0.0,NaN,417,0.364224,113
3,2018,19016,Nuevo León,Doctor González,539,0.559911,Developing industrial market,Broad B2B gap,0.559911,0.0,0.022717,0.0,0.0,NaN,424,0.361635,115
4,2018,20328,Oaxaca,San Pedro Taviche,549,0.556347,Developing industrial market,Broad B2B gap,0.556347,0.0,0.022717,0.0,0.0,NaN,431,0.359334,118
5,2018,20313,Oaxaca,San Pedro Jocotipac,634,0.522049,Developing industrial market,Broad B2B gap,0.522049,0.0,0.022717,0.0,0.0,NaN,493,0.337181,141
6,2018,21193,Puebla,Tzicatlacoyan,637,0.519822,Developing industrial market,Broad B2B gap,0.519822,0.0,0.022717,0.0,0.0,NaN,496,0.335742,141
7,2018,20564,Oaxaca,Yutanduchi de Guerrero,654,0.513586,Developing industrial market,Broad B2B gap,0.513586,0.0,0.022717,0.0,0.0,NaN,509,0.331715,145
8,2023,24059,San Luis Potosí,Villa de Pozos,2,0.957987,Top priority opportunity,Broad B2B gap,0.957987,0.0,0.028665,0.0,0.0,NaN,2,0.630762,0
9,2023,25019,Sinaloa,Eldorado,135,0.762801,Priority opportunity,Broad B2B gap,0.762801,0.0,0.028665,0.0,0.0,NaN,122,0.502247,13



VALIDACIÓN RÁPIDA
Filas diagnosticadas: 4530
Missing scaled_b2b_supply_percentile_v2: 0
Missing industrial_b2b_opportunity_score_v2: 0
Años: [np.int64(2018), np.int64(2023)]


In [ ]:
# ============================================================
# 10. CONSOLIDAR BASE FINAL DE SCORES DE OPORTUNIDAD
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
#
# Output folder:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/
#
# Archivos:
# 01_municipality_industrial_b2b_opportunity_scores.csv
# 01_municipality_industrial_b2b_opportunity_scores.parquet
# 02_top_opportunity_ranking.csv
# 03_opportunity_category_summary.csv
# 04_opportunity_subtype_summary.csv
# 05_scoring_methodology_notes.txt
#
# Score definitivo:
# industrial_b2b_opportunity_score =
# industrial_demand_percentile * (1 - scaled_b2b_supply_percentile)
#
# Ajuste V2:
# Si b2b_scaled_capacity_proxy == 0,
# entonces scaled_b2b_supply_percentile = 0.
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

modeling_path = (
    project_path / "data" / "processed" / "modeling_base"
)

final_scores_path = modeling_path / "final_scores"
final_scores_path.mkdir(parents=True, exist_ok=True)

input_path = modeling_path / "08_modeling_base_complete_cases.csv"

output_csv_path = (
    final_scores_path / "01_municipality_industrial_b2b_opportunity_scores.csv"
)

output_parquet_path = (
    final_scores_path / "01_municipality_industrial_b2b_opportunity_scores.parquet"
)

top_ranking_path = (
    final_scores_path / "02_top_opportunity_ranking.csv"
)

category_summary_path = (
    final_scores_path / "03_opportunity_category_summary.csv"
)

subtype_summary_path = (
    final_scores_path / "04_opportunity_subtype_summary.csv"
)

methodology_notes_path = (
    final_scores_path / "05_scoring_methodology_notes.txt"
)

print("Input path:")
print(input_path)

print("\nOutput folder:")
print(final_scores_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré el input en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base complete-case
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir variables y pesos
# ------------------------------------------------------------

demand_features = [
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment"
]

demand_weights = {
    "total_manufacturing_establishments": 0.25,
    "total_manufacturing_employment": 0.20,
    "total_manufacturing_value_added": 0.25,
    "total_manufacturing_income": 0.20,
    "total_manufacturing_investment": 0.10
}

b2b_supply_features = [
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes"
]

b2b_supply_weights = {
    "denue_logistics_storage_establishments": 0.35,
    "denue_professional_technical_establishments": 0.25,
    "denue_business_support_establishments": 0.20,
    "denue_b2b_support_scian_classes": 0.20
}

scaled_features = [
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_support_establishments"
]

required_cols = (
    [
        "year",
        "entidad_id",
        "entidad_name",
        "municipio_id",
        "municipio_name",
        "geo_key"
    ]
    + demand_features
    + b2b_supply_features
    + scaled_features
)

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas necesarias: {missing_cols}")

if not np.isclose(sum(demand_weights.values()), 1.0):
    raise ValueError("Los pesos de demanda no suman 1.")

if not np.isclose(sum(b2b_supply_weights.values()), 1.0):
    raise ValueError("Los pesos de oferta B2B no suman 1.")


# ------------------------------------------------------------
# 4. Convertir variables a numéricas
# ------------------------------------------------------------

numeric_cols = list(set(demand_features + b2b_supply_features + scaled_features))

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_numeric = df[numeric_cols].isna().sum()

print("\nMissing en variables usadas:")
print(missing_numeric[missing_numeric > 0])

if missing_numeric.sum() > 0:
    raise ValueError(
        "Hay missing en variables necesarias. "
        "Revisa que estés usando 08_modeling_base_complete_cases.csv."
    )


# ------------------------------------------------------------
# 5. Industrial demand score
# ------------------------------------------------------------

for col in demand_features:
    percentile_col = f"{col}_demand_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["industrial_demand_score"] = 0.0

for col, weight in demand_weights.items():
    df["industrial_demand_score"] += (
        df[f"{col}_demand_percentile"] * weight
    )

df["industrial_demand_percentile"] = (
    df
    .groupby("year")["industrial_demand_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 6. B2B supply score general
# ------------------------------------------------------------

for col in b2b_supply_features:
    percentile_col = f"{col}_b2b_supply_percentile"

    df[percentile_col] = (
        df
        .groupby("year")[col]
        .rank(pct=True, method="average")
    )

df["b2b_supply_score"] = 0.0

for col, weight in b2b_supply_weights.items():
    df["b2b_supply_score"] += (
        df[f"{col}_b2b_supply_percentile"] * weight
    )

df["b2b_supply_percentile"] = (
    df
    .groupby("year")["b2b_supply_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 7. Scaled B2B supply score V2
# ------------------------------------------------------------

medium_expected_workers = 65.5
large_expected_workers = 150

df["b2b_scaled_capacity_proxy"] = (
    medium_expected_workers * df["denue_b2b_medium_establishments"]
    + large_expected_workers * df["denue_b2b_large_establishments"]
)

df["has_scaled_b2b_supply"] = (
    df["b2b_scaled_capacity_proxy"] > 0
).astype(int)

df["scaled_b2b_supply_percentile"] = np.nan

for year_value, idx in df.groupby("year").groups.items():

    year_mask = df.index.isin(idx)
    positive_mask = year_mask & (df["b2b_scaled_capacity_proxy"] > 0)
    zero_mask = year_mask & (df["b2b_scaled_capacity_proxy"] == 0)

    df.loc[zero_mask, "scaled_b2b_supply_percentile"] = 0.0

    df.loc[positive_mask, "scaled_b2b_supply_percentile"] = (
        df.loc[positive_mask, "b2b_scaled_capacity_proxy"]
        .rank(pct=True, method="average")
    )

df["scaled_b2b_supply_score"] = df["scaled_b2b_supply_percentile"]


# ------------------------------------------------------------
# 8. Variables auxiliares de atomización
# ------------------------------------------------------------

df["b2b_micro_small_establishments"] = (
    df["denue_b2b_micro_establishments"]
    + df["denue_b2b_small_establishments"]
)

df["b2b_micro_small_share"] = np.where(
    df["denue_b2b_support_establishments"] > 0,
    df["b2b_micro_small_establishments"] / df["denue_b2b_support_establishments"],
    np.nan
)

df["only_micro_small_b2b_supply"] = (
    (df["denue_b2b_support_establishments"] > 0) &
    (df["denue_b2b_medium_large_establishments"] == 0)
).astype(int)


# ------------------------------------------------------------
# 9. Score definitivo de oportunidad
# ------------------------------------------------------------

df["industrial_b2b_opportunity_score"] = (
    df["industrial_demand_percentile"]
    * (1 - df["scaled_b2b_supply_percentile"])
)

df["industrial_b2b_opportunity_percentile"] = (
    df
    .groupby("year")["industrial_b2b_opportunity_score"]
    .rank(pct=True, method="average")
)


# ------------------------------------------------------------
# 10. Categorías interpretativas
# ------------------------------------------------------------

def classify_opportunity(row):
    demand_p = row["industrial_demand_percentile"]
    scaled_p = row["scaled_b2b_supply_percentile"]

    if demand_p >= 0.90 and scaled_p < 0.50:
        return "Top priority opportunity"

    elif demand_p >= 0.75 and scaled_p < 0.50:
        return "Priority opportunity"

    elif demand_p >= 0.75 and scaled_p >= 0.75:
        return "Industrial ecosystem hub"

    elif demand_p >= 0.50 and scaled_p < 0.75:
        return "Developing industrial market"

    elif demand_p < 0.50 and scaled_p >= 0.75:
        return "Service hub, lower industrial demand"

    elif demand_p < 0.50 and scaled_p < 0.75:
        return "Lower initial priority"

    else:
        return "Intermediate / monitor"


df["industrial_b2b_opportunity_category"] = (
    df.apply(classify_opportunity, axis=1)
)


# ------------------------------------------------------------
# 11. Subtipos de oportunidad
# ------------------------------------------------------------

def classify_opportunity_subtype(row):
    category = row["industrial_b2b_opportunity_category"]
    b2b_general_p = row["b2b_supply_percentile"]
    b2b_total = row["denue_b2b_support_establishments"]
    medium_large = row["denue_b2b_medium_large_establishments"]
    scaled_p = row["scaled_b2b_supply_percentile"]

    opportunity_categories = [
        "Top priority opportunity",
        "Priority opportunity",
        "Developing industrial market"
    ]

    if category not in opportunity_categories:
        return "Not opportunity subtype"

    if medium_large == 0 and b2b_total > 0 and b2b_general_p >= 0.75:
        return "Atomized B2B opportunity"

    elif medium_large == 0 and b2b_total > 0 and b2b_general_p < 0.75:
        return "Limited scaled B2B opportunity"

    elif medium_large == 0 and b2b_total == 0:
        return "Broad B2B gap"

    elif medium_large > 0 and scaled_p < 0.50:
        return "Low scaled B2B opportunity"

    else:
        return "Mixed opportunity case"


df["industrial_b2b_opportunity_subtype"] = (
    df.apply(classify_opportunity_subtype, axis=1)
)


# ------------------------------------------------------------
# 12. Ranking por año
# ------------------------------------------------------------

df = (
    df
    .sort_values(
        ["year", "industrial_b2b_opportunity_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

df["industrial_b2b_opportunity_rank"] = (
    df
    .groupby("year")
    .cumcount() + 1
)


# ------------------------------------------------------------
# 13. Seleccionar columnas finales
# ------------------------------------------------------------

final_cols = [
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",

    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    "b2b_micro_small_establishments",
    "b2b_micro_small_share",
    "only_micro_small_b2b_supply",
    "has_scaled_b2b_supply",
    "b2b_scaled_capacity_proxy",

    "industrial_demand_score",
    "industrial_demand_percentile",
    "b2b_supply_score",
    "b2b_supply_percentile",
    "scaled_b2b_supply_score",
    "scaled_b2b_supply_percentile",
    "industrial_b2b_opportunity_score",
    "industrial_b2b_opportunity_percentile",
    "industrial_b2b_opportunity_rank",

    "industrial_b2b_opportunity_category",
    "industrial_b2b_opportunity_subtype"
]

final_cols = [col for col in final_cols if col in df.columns]

final_scores = df[final_cols].copy()


# ------------------------------------------------------------
# 14. Resúmenes finales
# ------------------------------------------------------------

category_summary = (
    final_scores
    .groupby(["year", "industrial_b2b_opportunity_category"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        avg_opportunity_score=("industrial_b2b_opportunity_score", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_scaled_b2b_supply_percentile=("scaled_b2b_supply_percentile", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_manufacturing_establishments=("total_manufacturing_establishments", "mean"),
        avg_manufacturing_employment=("total_manufacturing_employment", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_micro_small_share=("b2b_micro_small_share", "mean")
    )
    .reset_index()
)

category_summary["share_municipality_years"] = (
    category_summary["n_municipality_years"]
    / category_summary.groupby("year")["n_municipality_years"].transform("sum")
)

subtype_summary = (
    final_scores
    .groupby(["year", "industrial_b2b_opportunity_subtype"], dropna=False)
    .agg(
        n_municipality_years=("geo_key", "size"),
        avg_opportunity_score=("industrial_b2b_opportunity_score", "mean"),
        avg_industrial_demand_percentile=("industrial_demand_percentile", "mean"),
        avg_scaled_b2b_supply_percentile=("scaled_b2b_supply_percentile", "mean"),
        avg_b2b_supply_percentile=("b2b_supply_percentile", "mean"),
        avg_b2b_total=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean")
    )
    .reset_index()
)

top_opportunity_ranking = (
    final_scores
    .sort_values(
        ["year", "industrial_b2b_opportunity_rank"],
        ascending=[True, True]
    )
    .groupby("year")
    .head(100)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 15. Guardar archivos finales
# ------------------------------------------------------------

final_scores.to_csv(
    output_csv_path,
    index=False,
    encoding="utf-8-sig"
)

try:
    final_scores.to_parquet(
        output_parquet_path,
        index=False
    )
except Exception as e:
    print("\nNo se pudo guardar parquet. El CSV sí fue guardado.")
    print(e)

top_opportunity_ranking.to_csv(
    top_ranking_path,
    index=False,
    encoding="utf-8-sig"
)

category_summary.to_csv(
    category_summary_path,
    index=False,
    encoding="utf-8-sig"
)

subtype_summary.to_csv(
    subtype_summary_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 16. Guardar nota metodológica
# ------------------------------------------------------------

methodology_notes = f"""
Nearshoring Project / Market Opportunity Analytics
Final scoring methodology notes

Input:
{input_path}

Output:
{output_csv_path}

Industrial demand score:
- Constructed from annual percentile ranks of:
  1. total_manufacturing_establishments, weight 0.25
  2. total_manufacturing_employment, weight 0.20
  3. total_manufacturing_value_added, weight 0.25
  4. total_manufacturing_income, weight 0.20
  5. total_manufacturing_investment, weight 0.10

B2B supply score:
- Constructed from annual percentile ranks of:
  1. denue_logistics_storage_establishments, weight 0.35
  2. denue_professional_technical_establishments, weight 0.25
  3. denue_business_support_establishments, weight 0.20
  4. denue_b2b_support_scian_classes, weight 0.20
- denue_b2b_support_establishments is not included in the score to avoid double-counting its components.

Scaled B2B supply score:
- b2b_scaled_capacity_proxy =
  {medium_expected_workers} * denue_b2b_medium_establishments
  + {large_expected_workers} * denue_b2b_large_establishments
- If b2b_scaled_capacity_proxy == 0, scaled_b2b_supply_percentile = 0.
- If b2b_scaled_capacity_proxy > 0, scaled_b2b_supply_percentile is calculated as the annual percentile rank among municipalities with positive scaled B2B capacity.

Final opportunity score:
industrial_b2b_opportunity_score =
industrial_demand_percentile * (1 - scaled_b2b_supply_percentile)

Interpretation:
- Higher scores indicate municipalities with stronger industrial demand and weaker scaled B2B supply.
- Categories and subtypes are interpretive labels used to distinguish:
  1. Atomized B2B opportunity
  2. Limited scaled B2B opportunity
  3. Broad B2B gap
  4. Industrial ecosystem hub
  5. Lower initial priority

Important scope note:
- This is a territorial prioritization model, not a causal prediction of nearshoring.
"""

with open(methodology_notes_path, "w", encoding="utf-8") as f:
    f.write(methodology_notes)


# ------------------------------------------------------------
# 17. Validación rápida
# ------------------------------------------------------------

print("\nPROCESO TERMINADO")
print("=" * 100)

print("\nBase final:")
print(f"Filas: {final_scores.shape[0]:,}")
print(f"Columnas: {final_scores.shape[1]:,}")

print("\nMissing principales:")
print(
    final_scores[
        [
            "industrial_demand_percentile",
            "b2b_supply_percentile",
            "scaled_b2b_supply_percentile",
            "industrial_b2b_opportunity_score",
            "industrial_b2b_opportunity_category",
            "industrial_b2b_opportunity_subtype"
        ]
    ]
    .isna()
    .sum()
)

print("\nAños:")
print(sorted(final_scores["year"].dropna().unique()))

print("\nTop 20 ranking final:")
display(top_opportunity_ranking.head(20))

print("\nResumen por categoría:")
display(category_summary)

print("\nResumen por subtipo:")
display(subtype_summary)

print("\nArchivos guardados:")
print(output_csv_path)
print(output_parquet_path)
print(top_ranking_path)
print(category_summary_path)
print(subtype_summary_path)
print(methodology_notes_path)

Google Drive ya está disponible en: /content/drive/MyDrive
Input path:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/08_modeling_base_complete_cases.csv

Output folder:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores

Base cargada:
Filas: 4,530
Columnas: 54

Missing en variables usadas:
Series([], dtype: int64)

PROCESO TERMINADO

Base final:
Filas: 4,530
Columnas: 37

Missing principales:
industrial_demand_percentile           0
b2b_supply_percentile                  0
scaled_b2b_supply_percentile           0
industrial_b2b_opportunity_score       0
industrial_b2b_opportunity_category    0
industrial_b2b_opportunity_subtype     0
dtype: int64

Años:
[np.int64(2018), np.int64(2023)]

Top 20 ranking final:


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,total_manufacturing_establishments,total_manufacturing_employment,total_manufacturing_value_added,total_manufacturing_income,...,industrial_demand_percentile,b2b_supply_score,b2b_supply_percentile,scaled_b2b_supply_score,scaled_b2b_supply_percentile,industrial_b2b_opportunity_score,industrial_b2b_opportunity_percentile,industrial_b2b_opportunity_rank,industrial_b2b_opportunity_category,industrial_b2b_opportunity_subtype
0,2018,15,México,101,Tianguistenco,15101,671.0,10225.0,23738.590,43770.764,...,0.968374,0.806013,0.814699,0.0,0.0,0.968374,1.000000,1,Top priority opportunity,Atomized B2B opportunity
1,2018,29,Tlaxcala,013,Huamantla,29013,743.0,7179.0,4539.760,14402.339,...,0.956793,0.879866,0.891314,0.0,0.0,0.956793,0.999555,2,Top priority opportunity,Atomized B2B opportunity
2,2018,14,Jalisco,008,Arandas,14008,458.0,6485.0,2100.558,7795.481,...,0.936303,0.846247,0.854343,0.0,0.0,0.936303,0.999109,3,Top priority opportunity,Atomized B2B opportunity
3,2018,11,Guanajuato,004,Apaseo el Alto,11004,404.0,3985.0,1912.168,6117.481,...,0.918486,0.853207,0.861024,0.0,0.0,0.918486,0.998664,4,Top priority opportunity,Atomized B2B opportunity
4,2018,31,Yucatán,041,Kanasín,31041,318.0,3541.0,1274.774,4645.405,...,0.909131,0.751269,0.766592,0.0,0.0,0.909131,0.998218,5,Top priority opportunity,Atomized B2B opportunity
5,2018,29,Tlaxcala,018,Contla de Juan Cuamatzi,29018,644.0,3937.0,386.251,1370.414,...,0.903786,0.686448,0.703786,0.0,0.0,0.903786,0.997773,6,Top priority opportunity,Limited scaled B2B opportunity
6,2018,13,Hidalgo,081,Zacualtipán de Ángeles,13081,338.0,3456.0,651.961,2668.787,...,0.899332,0.752116,0.767929,0.0,0.0,0.899332,0.997327,7,Priority opportunity,Atomized B2B opportunity
7,2018,31,Yucatán,038,Hunucmá,31038,257.0,1756.0,1692.994,4333.701,...,0.897550,0.659878,0.678396,0.0,0.0,0.897550,0.996882,8,Priority opportunity,Limited scaled B2B opportunity
8,2018,14,Jalisco,083,Tala,14083,249.0,3568.0,881.091,3958.324,...,0.896659,0.843630,0.852116,0.0,0.0,0.896659,0.996437,9,Priority opportunity,Atomized B2B opportunity
9,2018,04,Campeche,001,Calkiní,04001,1147.0,3999.0,337.851,476.206,...,0.893987,0.787661,0.797773,0.0,0.0,0.893987,0.995991,10,Priority opportunity,Atomized B2B opportunity



Resumen por categoría:


,year,industrial_b2b_opportunity_category,n_municipality_years,avg_opportunity_score,avg_industrial_demand_percentile,avg_scaled_b2b_supply_percentile,avg_b2b_supply_percentile,avg_manufacturing_establishments,avg_manufacturing_employment,avg_b2b_total,avg_b2b_medium_large,avg_micro_small_share,share_municipality_years
0,2018,Developing industrial market,672,0.516700,0.668647,0.193617,0.637374,243.049107,1363.879464,59.858631,1.337798,0.979746,0.299332
1,2018,Industrial ecosystem hub,162,0.114277,0.944598,0.877113,0.953878,1565.012346,30009.592593,1086.413580,76.469136,0.927062,0.072160
2,2018,Intermediate / monitor,2,0.165936,0.710579,0.766387,0.910913,295.500000,1458.000000,177.500000,11.000000,0.938024,0.000891
3,2018,Lower initial priority,1122,0.245738,0.250111,0.012481,0.291417,36.622103,77.178253,6.899287,0.072193,0.991980,0.499777
4,2018,Priority opportunity,258,0.710744,0.821189,0.132530,0.723372,377.321705,1761.395349,72.275194,0.658915,0.989785,0.114922
5,2018,Top priority opportunity,29,0.713418,0.921780,0.226477,0.852915,794.310345,5660.862069,176.068966,1.310345,0.990035,0.012918
6,2023,Developing industrial market,701,0.508602,0.670168,0.208940,0.645911,270.223966,1298.005706,55.748930,1.427960,0.975845,0.306783
7,2023,Industrial ecosystem hub,184,0.117460,0.942461,0.873508,0.945881,1534.614130,30323.554348,897.070652,78.103261,0.910977,0.080525
8,2023,Intermediate / monitor,1,0.149911,0.744420,0.798621,0.920350,216.000000,710.000000,175.000000,14.000000,0.920000,0.000438
9,2023,Lower initial priority,1142,0.243359,0.250109,0.018300,0.289694,42.126970,88.134851,6.204028,0.091068,0.986549,0.499781



Resumen por subtipo:


,year,industrial_b2b_opportunity_subtype,n_municipality_years,avg_opportunity_score,avg_industrial_demand_percentile,avg_scaled_b2b_supply_percentile,avg_b2b_supply_percentile,avg_b2b_total,avg_b2b_medium_large
0,2018,Atomized B2B opportunity,96,0.753937,0.753937,0.000000,0.802566,80.760417,0.000000
1,2018,Broad B2B gap,8,0.549499,0.549499,0.000000,0.022717,0.000000,0.000000
2,2018,Limited scaled B2B opportunity,429,0.653562,0.653562,0.000000,0.525629,22.023310,0.000000
3,2018,Low scaled B2B opportunity,272,0.543410,0.748070,0.270120,0.755855,80.433824,1.411765
4,2018,Mixed opportunity case,154,0.300803,0.826635,0.632459,0.853018,161.681818,4.694805
5,2018,Not opportunity subtype,1286,0.229054,0.338313,0.122573,0.375832,143.153188,9.713064
6,2023,Atomized B2B opportunity,84,0.761954,0.761954,0.000000,0.801881,71.857143,0.000000
7,2023,Broad B2B gap,7,0.700594,0.700594,0.000000,0.028665,0.000000,0.000000
8,2023,Limited scaled B2B opportunity,408,0.651834,0.651834,0.000000,0.520696,19.247549,0.000000
9,2023,Low scaled B2B opportunity,288,0.535779,0.724807,0.256590,0.738836,63.965278,1.361111



Archivos guardados:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/01_municipality_industrial_b2b_opportunity_scores.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/01_municipality_industrial_b2b_opportunity_scores.parquet
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/02_top_opportunity_ranking.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/03_opportunity_category_summary.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/04_opportunity_subtype_summary.csv
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/05_scoring_methodology_notes.txt


In [1]:
# ============================================================
# GENERAR MUESTRA CSV DE LA FINAL OPPORTUNITY SCORE BASE
# Nearshoring Project / Market Opportunity Analytics
#
# Funciona aunque hayas cerrado sesión.
#
# Objetivo:
# Crear una muestra pequeña en .csv para compartir en otro chat.
#
# Input:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/
#   01_municipality_industrial_b2b_opportunity_scores.csv
#
# Output:
# /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/context_exports/
#   final_opportunity_score_base_sample_for_context.csv
# ============================================================


# ------------------------------------------------------------
# 0. Imports y montaje robusto de Google Drive
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np


def get_drive_root():
    possible_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/gdrive/MyDrive")
    ]

    for root in possible_roots:
        if root.exists():
            print(f"Google Drive ya está disponible en: {root}")
            return root

    try:
        from google.colab import drive
        print("Montando Google Drive en /content/drive...")
        drive.mount("/content/drive")

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("Primer intento de montaje falló:")
        print(e)

    try:
        from google.colab import drive
        print("Intentando force_remount=True...")
        drive.mount("/content/drive", force_remount=True)

        root = Path("/content/drive/MyDrive")
        if root.exists():
            print(f"Google Drive montado correctamente en: {root}")
            return root

    except Exception as e:
        print("El force_remount también falló:")
        print(e)

    raise RuntimeError(
        "No se pudo montar Google Drive. "
        "Reinicia el runtime y vuelve a autorizar Drive."
    )


drive_root = get_drive_root()


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

final_scores_path = (
    project_path / "data" / "processed" / "modeling_base" / "final_scores"
)

input_path = final_scores_path / "01_municipality_industrial_b2b_opportunity_scores.csv"

output_path = final_scores_path / "context_exports"
output_path.mkdir(parents=True, exist_ok=True)

sample_csv_path = output_path / "final_opportunity_score_base_sample_for_context.csv"

print("Input:")
print(input_path)

print("\nOutput:")
print(sample_csv_path)

if not input_path.exists():
    raise FileNotFoundError(f"No encontré la base final en: {input_path}")


# ------------------------------------------------------------
# 2. Cargar base final
# ------------------------------------------------------------

df = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

df["entidad_id"] = (
    df["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

df["municipio_id"] = (
    df["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

df["geo_key"] = df["entidad_id"] + df["municipio_id"]

print("\nBase cargada:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
print("Años:", sorted(df["year"].dropna().unique()))


# ------------------------------------------------------------
# 3. Columnas que sí conviene mostrar en la muestra
# ------------------------------------------------------------

sample_cols = [
    # IDs
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    # Ranking y clasificación
    "industrial_b2b_opportunity_rank",
    "industrial_b2b_opportunity_category",
    "industrial_b2b_opportunity_subtype",

    # Scores principales
    "industrial_b2b_opportunity_score",
    "industrial_b2b_opportunity_percentile",
    "industrial_demand_percentile",
    "b2b_supply_percentile",
    "scaled_b2b_supply_percentile",

    # Variables manufactureras clave
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",

    # Variables B2B clave
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    # Tamaño B2B
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    # Auxiliares
    "b2b_micro_small_share",
    "only_micro_small_b2b_supply",
    "has_scaled_b2b_supply",
    "b2b_scaled_capacity_proxy"
]

sample_cols = [col for col in sample_cols if col in df.columns]


# ------------------------------------------------------------
# 4. Validar columnas esenciales
# ------------------------------------------------------------

required_cols = [
    "year",
    "industrial_b2b_opportunity_rank",
    "industrial_b2b_opportunity_score",
    "industrial_b2b_opportunity_category",
    "industrial_b2b_opportunity_subtype"
]

missing_required = [col for col in required_cols if col not in df.columns]

if missing_required:
    raise ValueError(f"Faltan columnas esenciales: {missing_required}")


# ------------------------------------------------------------
# 5. Construir muestra útil para contexto
# ------------------------------------------------------------

# A) Top 15 por año
top_sample = (
    df[sample_cols]
    .sort_values(
        ["year", "industrial_b2b_opportunity_rank"],
        ascending=[True, True]
    )
    .groupby("year")
    .head(15)
    .copy()
)

top_sample["sample_source"] = "top_15_by_year"


# B) Hasta 2 ejemplos por categoría y año
category_sample = (
    df[sample_cols]
    .sort_values(
        ["year", "industrial_b2b_opportunity_category", "industrial_b2b_opportunity_score"],
        ascending=[True, True, False]
    )
    .groupby(["year", "industrial_b2b_opportunity_category"], dropna=False)
    .head(2)
    .copy()
)

category_sample["sample_source"] = "examples_by_category"


# C) Hasta 2 ejemplos por subtipo y año
subtype_sample = (
    df[sample_cols]
    .sort_values(
        ["year", "industrial_b2b_opportunity_subtype", "industrial_b2b_opportunity_score"],
        ascending=[True, True, False]
    )
    .groupby(["year", "industrial_b2b_opportunity_subtype"], dropna=False)
    .head(2)
    .copy()
)

subtype_sample["sample_source"] = "examples_by_subtype"


# D) Muestra aleatoria pequeña por año para contexto general
random_sample = (
    df[sample_cols]
    .groupby("year", group_keys=False)
    .apply(lambda x: x.sample(n=min(10, len(x)), random_state=123))
    .copy()
)

random_sample["sample_source"] = "random_sample_by_year"


# Unir y quitar duplicados
sample_df = pd.concat(
    [
        top_sample,
        category_sample,
        subtype_sample,
        random_sample
    ],
    ignore_index=True
)

sample_df = (
    sample_df
    .drop_duplicates(subset=["year", "geo_key"], keep="first")
    .sort_values(
        ["year", "industrial_b2b_opportunity_rank"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Redondear columnas numéricas para que sea legible
# ------------------------------------------------------------

score_like_cols = [
    col for col in sample_df.columns
    if (
        "score" in col
        or "percentile" in col
        or "share" in col
        or "proxy" in col
    )
]

for col in score_like_cols:
    sample_df[col] = pd.to_numeric(sample_df[col], errors="ignore")
    if pd.api.types.is_numeric_dtype(sample_df[col]):
        sample_df[col] = sample_df[col].round(4)


# ------------------------------------------------------------
# 7. Guardar muestra CSV
# ------------------------------------------------------------

sample_df.to_csv(
    sample_csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 8. Mostrar resultado
# ------------------------------------------------------------

print("\nMUESTRA GENERADA")
print("=" * 100)
print(f"Filas de muestra: {sample_df.shape[0]:,}")
print(f"Columnas de muestra: {sample_df.shape[1]:,}")

print("\nArchivo guardado en:")
print(sample_csv_path)

print("\nVista rápida:")
display(sample_df.head(30))

Montando Google Drive en /content/drive...
Mounted at /content/drive
Google Drive montado correctamente en: /content/drive/MyDrive
Input:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/01_municipality_industrial_b2b_opportunity_scores.csv

Output:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/context_exports/final_opportunity_score_base_sample_for_context.csv

Base cargada:
Filas: 4,530
Columnas: 37
Años: [np.int64(2018), np.int64(2023)]


/tmp/ipykernel_5038/4200677502.py:270: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(10, len(x)), random_state=123))
/tmp/ipykernel_5038/4200677502.py:314: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  sample_df[col] = pd.to_numeric(sample_df[col], errors="ignore")



MUESTRA GENERADA
Filas de muestra: 76
Columnas de muestra: 34

Archivo guardado en:
/content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/context_exports/final_opportunity_score_base_sample_for_context.csv

Vista rápida:


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_b2b_opportunity_rank,industrial_b2b_opportunity_category,industrial_b2b_opportunity_subtype,industrial_b2b_opportunity_score,...,denue_b2b_micro_establishments,denue_b2b_small_establishments,denue_b2b_medium_establishments,denue_b2b_large_establishments,denue_b2b_medium_large_establishments,b2b_micro_small_share,only_micro_small_b2b_supply,has_scaled_b2b_supply,b2b_scaled_capacity_proxy,sample_source
0,2018,15,México,101,Tianguistenco,15101,1,Top priority opportunity,Atomized B2B opportunity,0.9684,...,159.0,8.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
1,2018,29,Tlaxcala,013,Huamantla,29013,2,Top priority opportunity,Atomized B2B opportunity,0.9568,...,163.0,11.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
2,2018,14,Jalisco,008,Arandas,14008,3,Top priority opportunity,Atomized B2B opportunity,0.9363,...,176.0,11.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
3,2018,11,Guanajuato,004,Apaseo el Alto,11004,4,Top priority opportunity,Atomized B2B opportunity,0.9185,...,106.0,5.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
4,2018,31,Yucatán,041,Kanasín,31041,5,Top priority opportunity,Atomized B2B opportunity,0.9091,...,49.0,8.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
5,2018,29,Tlaxcala,018,Contla de Juan Cuamatzi,29018,6,Top priority opportunity,Limited scaled B2B opportunity,0.9038,...,75.0,1.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
6,2018,13,Hidalgo,081,Zacualtipán de Ángeles,13081,7,Priority opportunity,Atomized B2B opportunity,0.8993,...,68.0,3.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
7,2018,31,Yucatán,038,Hunucmá,31038,8,Priority opportunity,Limited scaled B2B opportunity,0.8976,...,40.0,1.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
8,2018,14,Jalisco,083,Tala,14083,9,Priority opportunity,Atomized B2B opportunity,0.8967,...,97.0,12.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
9,2018,04,Campeche,001,Calkiní,04001,10,Priority opportunity,Atomized B2B opportunity,0.8940,...,70.0,3.0,0.0,0.0,0.0,1.0000,1,0,0.0,top_15_by_year
